# 🇺🇸 A Century of American Carbon — standalone edition
### A data story you can run anywhere (no setup, no downloads)

This notebook is **fully self-contained**: the plotting library and every dataset are embedded below. Nothing to clone, install from GitHub, or upload.

**To present:** run the three Setup cells once (or `Runtime ▸ Run all`), then step through the graph cells one at a time as you talk.

## Setup — run these three cells first

In [ ]:
!pip -q install matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print('deps ready ✓')

### The `viz_lib` plotting functions (embedded)

In [ ]:
"""Shared visual identity for every plot in the library.

One place sets the palette, fonts, grid and spines so that a scatter plot and
a line plot drawn a year apart still look like they came from the same tool.

The categorical palette is the validated, colorblind-safe order from the
project's data-viz design system (worst adjacent CVD deltaE ~9, normal-vision
~20). Hues are assigned to series in fixed order and never cycled: past eight
series, fold the tail into "Other" or facet instead of inventing new colors.
"""


import matplotlib as mpl

#: Sequential "smog / heat" ramp for CO2 magnitude: pale haze -> deep ember.
#: Lightness decreases monotonically, so it stays legible in black & white and
#: for colorblind readers (darker always means more emissions).
SMOG: list[str] = ["#f4d06a", "#eaa23b", "#df6b2e", "#c0392b", "#7b1f16"]


def smog_color(value: float, vmax: float) -> tuple:
    """Map ``value`` (0..``vmax``) onto the SMOG ramp; returns an RGBA tuple.

    Colors magnitude, not identity: hotter = more CO2 per capita. Pass the same
    ``vmax`` to several charts so their colors share one honest scale.
    """
    import matplotlib.colors as mcolors

    cmap = mcolors.LinearSegmentedColormap.from_list("smog", SMOG)
    frac = 0.0 if vmax <= 0 else max(0.0, min(1.0, value / vmax))
    # start a little into the ramp so the smallest bars aren't near-white
    return cmap(0.12 + 0.88 * frac)


#: Categorical palette, light surface, in fixed assignment order.
PALETTE: list[str] = [
    "#2a78d6",  # 1 blue
    "#eb6834",  # 2 orange
    "#1baf7a",  # 3 aqua
    "#eda100",  # 4 yellow
    "#e87ba4",  # 5 magenta
    "#008300",  # 6 green
    "#4a3aa7",  # 7 violet
    "#e34948",  # 8 red
]

# Chrome / ink for the light surface these plots render on.
_INK_PRIMARY = "#0b0b0b"
_INK_SECONDARY = "#52514e"
_MUTED = "#898781"
_GRID = "#e1e0d9"
_AXIS = "#c3c2b7"
_SURFACE = "#fcfcfb"

_FONT_STACK = ["system-ui", "Segoe UI", "DejaVu Sans", "Arial", "sans-serif"]


def series_color(index: int) -> str:
    """Return the palette hue for the *index*-th series (0-based).

    Raises ``IndexError`` past the eighth slot rather than silently cycling a
    hue — a reused color is a correctness bug in a categorical chart. Callers
    with more than eight series should fold the tail into "Other" or facet.
    """
    if index < 0 or index >= len(PALETTE):
        raise IndexError(
            f"series index {index} out of range; the categorical palette has "
            f"{len(PALETTE)} slots. Fold extra series into 'Other' or facet."
        )
    return PALETTE[index]


def apply_theme() -> None:
    """Apply the library's rcParams globally.

    Idempotent — safe to call more than once. Called automatically the first
    time a plot function runs, so most users never call it directly.
    """
    mpl.rcParams.update(
        {
            "figure.facecolor": _SURFACE,
            "axes.facecolor": _SURFACE,
            "savefig.facecolor": _SURFACE,
            "font.family": "sans-serif",
            "font.sans-serif": _FONT_STACK,
            "font.size": 11,
            "text.color": _INK_PRIMARY,
            "axes.edgecolor": _AXIS,
            "axes.labelcolor": _INK_SECONDARY,
            "axes.titlecolor": _INK_PRIMARY,
            "axes.titlesize": 12.5,
            "axes.titleweight": "bold",
            "axes.linewidth": 1.0,
            "axes.grid": True,
            "axes.grid.axis": "y",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "grid.color": _GRID,
            "grid.linewidth": 1.0,
            "xtick.color": _MUTED,
            "ytick.color": _MUTED,
            "xtick.labelsize": 10.5,
            "ytick.labelsize": 10.5,
            "axes.prop_cycle": mpl.cycler(color=PALETTE),
            "legend.frameon": False,
            "legend.fontsize": 10.5,
        }
    )


"""Ranked horizontal bar chart, built to the Evergreen Data Viz Checklist.

One job: take a tidy pandas DataFrame and draw a sorted, directly-labelled
horizontal bar chart that a non-technical reader understands at a glance.

Checklist choices baked in: bars sorted by value (not alphabetically), a zero
baseline, direct value labels, no gridlines / border / redundant axis, a
takeaway title, and a CO2-themed sequential color ramp that stays legible in
black & white and for colorblind readers.
"""


import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch


_UP, _DOWN = "▲", "▼"  # ▲ ▼
_SURFACE = "#fcfcfb"


def _readable_ink(color) -> str:
    """Pick black or white text for a filled segment by its luminance."""
    r, g, b = mcolors.to_rgb(color)
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    return "#0b0b0b" if lum > 0.6 else "#ffffff"


def ranked_bar(
    df,
    category: str,
    value: str,
    *,
    compare: str | None = None,
    reference: float | None = None,
    reference_label: str | None = None,
    vmax: float | None = None,
    unit: str = "",
    value_fmt: str = "{:.1f}",
    title: str | None = None,
    subtitle: str | None = None,
    note: str | None = None,
    ascending: bool = False,
    ax=None,
    figsize: tuple[float, float] | None = None,
):
    """Draw a sorted horizontal bar chart from a pandas DataFrame.

    Parameters
    ----------
    df
        A pandas DataFrame (MVP input — DataFrames only).
    category, value
        Column names: the label per bar and the numeric length to rank by.
    compare
        Optional column with an earlier value; when given, each bar gets a
        muted ``▲ / ▼ %`` tag showing the change to ``value`` (the story of
        who rose or fell).
    reference, reference_label
        Optional vertical reference line (e.g. the world average) and its
        label — instant context for "how far above normal".
    vmax
        Upper bound for the color ramp. Pass the same ``vmax`` to several
        charts so a bar of a given darkness means the same emissions in each.
        Defaults to this chart's maximum.
    unit
        Unit string appended to the reference label / used in labels.
    value_fmt
        Format string for the value labels.
    title, subtitle, note
        Takeaway title, units/what-am-I-looking-at subtitle, and a source note.
    ascending
        Sort direction; default puts the largest bar on top.
    ax, figsize
        Optional target Axes and figure size (height auto-scales with bars).

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_theme()

    data = df[[category, value] + ([compare] if compare else [])].dropna(subset=[value])
    data = data.sort_values(value, ascending=ascending).reset_index(drop=True)
    labels = data[category].tolist()
    values = data[value].tolist()
    n = len(values)
    if n == 0:
        raise ValueError("no rows to plot after dropping missing values")

    top = vmax if vmax is not None else max(values)

    owns_fig = ax is None
    if owns_fig:
        if figsize is None:
            figsize = (7.6, 0.52 * n + 1.7)
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # bars top-to-bottom (largest at top when ascending=False)
    y = list(range(n))[::-1]
    for yi, val in zip(y, values):
        ax.barh(yi, val, height=0.68, color=smog_color(val, top), zorder=3)

    xmax = max(values + ([reference] if reference else []))
    ax.set_xlim(0, xmax * 1.16)  # head-room for end labels

    # direct value labels (+ optional change tag) at each bar end
    for yi, (val, row) in zip(y, zip(values, data.itertuples(index=False))):
        label = value_fmt.format(val)
        ax.annotate(label, xy=(val, yi), xytext=(6, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=11, fontweight="bold",
                    color="#0b0b0b")
        if compare:
            prev = getattr(row, compare) if hasattr(row, compare) else None
            if prev:
                pct = (val - prev) / prev * 100
                up = pct >= 0
                tag = f"  {_UP if up else _DOWN} {abs(pct):.0f}%"
                # up = more emissions (bad) = warm; down = good = green
                ax.annotate(tag, xy=(val, yi), xytext=(6 + 34, 0),
                            textcoords="offset points", va="center", ha="left",
                            fontsize=9.5, fontweight="bold",
                            color="#c0392b" if up else "#0a7d33")

    # optional reference line for context, labelled at the baseline
    if reference is not None:
        ax.axvline(reference, color="#52514e", linestyle=(0, (4, 3)),
                   linewidth=1.2, zorder=2)
        rlab = reference_label or "reference"
        val_txt = f"{value_fmt.format(reference)}{(' ' + unit) if unit else ''}"
        ax.annotate(f"{rlab} ({val_txt})", xy=(reference, -0.75),
                    xytext=(4, 0), textcoords="offset points",
                    va="center", ha="left", fontsize=9.5, style="italic",
                    color="#52514e")

    # category labels; strip every non-data line (checklist: mute the lines)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=11)
    ax.set_xticks([])
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.grid(False)
    ax.tick_params(length=0)
    # headroom above the top bar for the subtitle, a little below for the ref label
    ax.set_ylim(-1.1, n - 1 + 0.9)

    # titles: takeaway on top, quiet subtitle beneath, source note at the foot
    if title:
        ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=24)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 8), textcoords="offset points",
                    ha="left", va="bottom", fontsize=11, color="#52514e")
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction",
                    xytext=(0, -26), textcoords="offset points",
                    ha="left", va="top", fontsize=8.5, color="#898781")

    if owns_fig:
        fig.tight_layout()
    return fig


def stacked_bar(
    df,
    category: str,
    segments: list[str],
    *,
    colors=None,
    ascending: bool = False,
    unit: str = "",
    value_fmt: str = "{:.0f}",
    seg_label_min: float = 0.08,
    title: str | None = None,
    subtitle: str | None = None,
    note: str | None = None,
    legend: bool = True,
    ax=None,
    figsize: tuple[float, float] | None = None,
):
    """Draw a ranked, stacked horizontal bar chart from a pandas DataFrame.

    Each row is one category (e.g. a country); ``segments`` are the columns
    that stack into its total. Rows are ordered by total, the total is labelled
    at the bar end, and wide-enough segments are labelled in place — the same
    idea as the Our World in Data "by source" charts.

    Parameters
    ----------
    df
        A pandas DataFrame, one row per category.
    category
        Column holding the row label per bar.
    segments
        Columns to stack, left-to-right. Missing values count as zero.
    colors
        ``None`` assigns the categorical palette in order (segments are
        categories — identity, not magnitude), or a dict of overrides.
    ascending
        Sort direction by total; default puts the largest total on top.
    unit, value_fmt
        Number format for the labels. ``value_fmt`` may be a format string
        (``unit`` is appended) or a callable ``value -> str`` for adaptive
        formatting (e.g. one decimal below 10, none above); with a callable,
        ``unit`` is ignored and the callable owns the whole label.
    seg_label_min
        Only label a segment in place if it is at least this fraction of the
        largest row total (keeps small slivers uncluttered).
    title, subtitle, note
        Takeaway title, quiet subtitle, and source note.
    legend
        Draw a segment legend above the plot (default True).
    ax, figsize
        Optional target Axes and figure size (height auto-scales with rows).

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_theme()

    data = df[[category] + segments].copy()
    for s in segments:
        data[s] = data[s].fillna(0.0)
    data["_total"] = data[segments].sum(axis=1)
    data = data.sort_values("_total", ascending=ascending).reset_index(drop=True)
    n = len(data)
    if n == 0:
        raise ValueError("no rows to plot")

    overrides = dict(colors) if isinstance(colors, dict) else {}
    seg_colors = {s: overrides.get(s, series_color(i)) for i, s in enumerate(segments)}

    owns_fig = ax is None
    if owns_fig:
        if figsize is None:
            figsize = (9.0, 0.55 * n + 2.0)
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    # one formatter for both the in-segment labels and the bar-end total;
    # pass a callable for adaptive formatting (e.g. "6.4 t" but "34 t")
    if callable(value_fmt):
        fmt = value_fmt
    else:
        def fmt(v):
            return f"{value_fmt.format(v)}{(' ' + unit) if unit else ''}"

    y = list(range(n))[::-1]
    max_total = float(data["_total"].max())
    label_floor = max_total * seg_label_min

    for yi, (_, row) in zip(y, data.iterrows()):
        left = 0.0
        for s in segments:
            w = float(row[s])
            if w <= 0:
                continue
            color = seg_colors[s]
            ax.barh(yi, w, left=left, height=0.7, color=color, zorder=3,
                    edgecolor=_SURFACE, linewidth=1.0)
            if w >= label_floor:  # label only segments wide enough to fit
                ax.annotate(fmt(w), xy=(left + w / 2, yi),
                            ha="center", va="center", fontsize=9.5,
                            fontweight="bold", color=_readable_ink(color), zorder=4)
            left += w
        # total at the bar end
        ax.annotate(fmt(left), xy=(left, yi), xytext=(6, 0),
                    textcoords="offset points", va="center", ha="left",
                    fontsize=11, fontweight="bold", color="#0b0b0b")

    ax.set_xlim(0, max_total * 1.16)
    ax.set_ylim(-0.8, n - 1 + (1.4 if legend else 0.7))
    ax.set_yticks(y)
    ax.set_yticklabels(data[category].tolist(), fontsize=11)
    ax.set_xticks([])
    for side in ("top", "right", "bottom", "left"):
        ax.spines[side].set_visible(False)
    ax.grid(False)
    ax.tick_params(length=0)

    if legend:
        handles = [Patch(facecolor=seg_colors[s], label=s) for s in segments]
        ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1.0),
                  ncol=min(len(segments), 6), frameon=False, fontsize=9.5,
                  handlelength=1.1, columnspacing=1.4, borderaxespad=0)

    if title:
        ax.set_title(title, loc="left", fontsize=15, fontweight="bold", pad=44)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 26), textcoords="offset points", ha="left",
                    va="bottom", fontsize=11, color="#52514e")
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction", xytext=(0, -26),
                    textcoords="offset points", ha="left", va="top",
                    fontsize=8.5, color="#898781")

    if owns_fig:
        fig.tight_layout()
    return fig


"""Stacked area chart — composition (part-to-whole) over time.

One job: show how several series stack into a total across an ordered x-axis,
with direct band labels and optional dated event markers for a narrative,
editorial "hero" chart.
"""


import numpy as np
import matplotlib.pyplot as plt


_MUTED = "#898781"
_INK = "#0b0b0b"
_SECOND = "#52514e"
_SURFACE = "#fcfcfb"


def stacked_area(
    df,
    x: str,
    series: list[str],
    *,
    colors=None,
    y_label: str | None = None,
    title: str | None = None,
    subtitle: str | None = None,
    note: str | None = None,
    events: list[dict] | None = None,
    direct_labels: bool = True,
    ax=None,
    figsize: tuple[float, float] | None = None,
):
    """Draw a stacked area chart from a pandas DataFrame.

    Parameters
    ----------
    df
        A pandas DataFrame.
    x
        Column name for the (ordered) x-axis, e.g. ``"Year"``.
    series
        Column names to stack, in **bottom-to-top** order. Missing values are
        treated as zero (so a band simply starts once its data begins).
    colors
        ``None`` assigns the categorical palette in order (fuels are
        categories, so identity — not magnitude — drives color), or pass a
        dict of ``{series_name: color}`` to override.
    y_label, title, subtitle, note
        Axis label, takeaway title, quiet subtitle, and source note.
    events
        Optional list of ``{"year": int, "label": str, "y": float}`` markers.
        ``y`` (0–1, default 0.95) sets the label height as a fraction of the
        axis; each draws a thin vertical rule + label for a dated annotation.
    direct_labels
        Label each band at its right end instead of a legend (default True).
    ax, figsize
        Optional target Axes and figure size.

    Returns
    -------
    matplotlib.figure.Figure
    """
    apply_theme()

    data = df.sort_values(x)
    xv = data[x].to_numpy(dtype=float)
    stacks = [np.nan_to_num(data[s].to_numpy(dtype=float), nan=0.0) for s in series]
    overrides = dict(colors) if isinstance(colors, dict) else {}
    cols = [overrides.get(s, series_color(i)) for i, s in enumerate(series)]

    owns_fig = ax is None
    if owns_fig:
        fig, ax = plt.subplots(figsize=figsize or (11, 6))
    else:
        fig = ax.figure

    # 2px surface gap between bands (checklist: separate the fills)
    ax.stackplot(xv, *stacks, colors=cols, edgecolor=_SURFACE, linewidth=0.8)

    ax.set_xlim(xv.min(), xv.max())
    top = np.sum(stacks, axis=0).max()
    ax.set_ylim(0, top * 1.02)

    # always mark the final year (e.g. 2024) as a tick, like the OWID charts
    ticks = [t for t in ax.get_xticks() if xv.min() <= t <= xv.max()]
    if not ticks or xv.max() - ticks[-1] > 6:
        ticks.append(xv.max())
    else:
        ticks[-1] = xv.max()  # snap a too-close tick onto the exact end
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"{int(t)}" for t in ticks])

    # direct band labels at the right end, nudged apart if they collide
    if direct_labels:
        cum = np.cumsum(stacks, axis=0)
        centers = []
        for i, s in enumerate(series):
            bottom = cum[i - 1][-1] if i else 0.0
            centers.append(((bottom + cum[i][-1]) / 2, s, cols[i]))
        _labels_at_right(ax, xv.max(), centers, top)

    # dated event markers (the editorial "timeline" layer)
    x_lo, x_hi = xv.min(), xv.max()
    near_right = x_lo + 0.85 * (x_hi - x_lo)
    for ev in events or []:
        yr = ev["year"]
        ax.axvline(yr, color=_MUTED, linewidth=1.0, linestyle=(0, (2, 2)), zorder=5)
        yfrac = ev.get("y", 0.95)
        # flip the label to the left of its rule near the right edge, so it
        # never lands on top of the right-hand band labels
        right = yr >= near_right
        ax.annotate(ev["label"], xy=(yr, top * yfrac),
                    xytext=(-4 if right else 4, 0), textcoords="offset points",
                    va="top", ha="right" if right else "left",
                    fontsize=9, color=_SECOND, zorder=6,
                    bbox=dict(boxstyle="round,pad=0.15", fc=_SURFACE, ec="none", alpha=0.85))

    # chrome: mute everything that isn't data (checklist: mute the lines)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.spines["left"].set_color(_MUTED)
    ax.spines["bottom"].set_color(_MUTED)
    ax.tick_params(length=0)
    ax.grid(False)
    if y_label:
        ax.set_ylabel(y_label, color=_SECOND)

    if title:
        ax.set_title(title, loc="left", fontsize=17, fontweight="bold", pad=26)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 8), textcoords="offset points", ha="left",
                    va="bottom", fontsize=11.5, color=_SECOND)
    if note:
        ax.annotate(note, xy=(0, 0), xycoords="axes fraction", xytext=(0, -34),
                    textcoords="offset points", ha="left", va="top",
                    fontsize=8.5, color=_MUTED)

    if owns_fig:
        fig.tight_layout()
    return fig


def _labels_at_right(ax, x_end, centers, top):
    """Place band labels at the right edge, spread so they don't overlap."""
    gap = top * 0.045
    centers = sorted(centers, key=lambda t: t[0])
    prev = None
    for y_val, name, color in centers:
        y_text = y_val if prev is None else max(y_val, prev + gap)
        prev = y_text
        ax.annotate(name, xy=(x_end, y_text), xytext=(8, 0),
                    textcoords="offset points", va="center", ha="left",
                    fontsize=10.5, fontweight="bold", color=color,
                    annotation_clip=False)

print('viz_lib functions defined ✓')

### The datasets (embedded)

In [ ]:
_DATA = {}
_DATA["co2_per_capita"] = (
"""RW50aXR5LENvZGUsWWVhcixDT+KCgiBlbWlzc2lvbnMgcGVyIGNhcGl0YQpBZmdoYW5pc3RhbixBRkcsMjAxNCwwLjI2NTIzMzIyCkFmZ2hhbmlzdGFuLEFG
RywyMDI0LDAuMjUzODQ4MzQKQWZyaWNhLE9XSURfQUZSLDIwMTQsMS4xNDYzMjA4CkFmcmljYSxPV0lEX0FGUiwyMDI0LDAuOTkzMjk1OTcKQWxiYW5pYSxB
TEIsMjAxNCwyLjA5MTI1OQpBbGJhbmlhLEFMQiwyMDI0LDEuNTkxOTkwMQpBbGdlcmlhLERaQSwyMDE0LDMuODg4MTM5NwpBbGdlcmlhLERaQSwyMDI0LDQu
MjMzODE3CkFuZG9ycmEsQU5ELDIwMTQsNi4yNTk0MjY2CkFuZG9ycmEsQU5ELDIwMjQsNS4xODE2NjA3CkFuZ29sYSxBR08sMjAxNCwwLjk2MDY1NTgKQW5n
b2xhLEFHTywyMDI0LDAuNTg5NDk2NwpBbmd1aWxsYSxBSUEsMjAxNCw4Ljc5NDg3OQpBbmd1aWxsYSxBSUEsMjAyNCwxMC4xMjY4NTUKQW50aWd1YSBhbmQg
QmFyYnVkYSxBVEcsMjAxNCw2LjM5NjY0OApBbnRpZ3VhIGFuZCBCYXJidWRhLEFURywyMDI0LDcuMDkyNDA2CkFyZ2VudGluYSxBUkcsMjAxNCw0LjM4MjE3
OTMKQXJnZW50aW5hLEFSRywyMDI0LDMuNzQzNDA2MwpBcm1lbmlhLEFSTSwyMDE0LDEuOTE3NjE5CkFybWVuaWEsQVJNLDIwMjQsMi40OTg2OTEKQXJ1YmEs
QUJXLDIwMTQsOC40MzUxMjgKQXJ1YmEsQUJXLDIwMjQsOC41MTg0ODcKQXNpYSxPV0lEX0FTSSwyMDE0LDQuMjc0NjkwNgpBc2lhLE9XSURfQVNJLDIwMjQs
NC44Njc4MTcKQXNpYSAoZXhjbC4gQ2hpbmEgYW5kIEluZGlhKSwsMjAxNCwzLjkzOTY3CkFzaWEgKGV4Y2wuIENoaW5hIGFuZCBJbmRpYSksLDIwMjQsNC4w
ODcyNDQKQXVzdHJhbGlhLEFVUywyMDE0LDE2LjYzOTEwMwpBdXN0cmFsaWEsQVVTLDIwMjQsMTQuNDc3MTk5CkF1c3RyaWEsQVVULDIwMTQsNy41MDU4NjY1
CkF1c3RyaWEsQVVULDIwMjQsNi4xODAxMTIKQXplcmJhaWphbixBWkUsMjAxNCwzLjU2NjAxMzMKQXplcmJhaWphbixBWkUsMjAyNCwzLjg1MzMyMDEKQmFo
YW1hcyxCSFMsMjAxNCw1LjM5NTY3NwpCYWhhbWFzLEJIUywyMDI0LDcuNjQ5NjgKQmFocmFpbixCSFIsMjAxNCwyMy41OTAzNDIKQmFocmFpbixCSFIsMjAy
NCwyNC4yNzAwODIKQmFuZ2xhZGVzaCxCR0QsMjAxNCwwLjQyNzYzODUKQmFuZ2xhZGVzaCxCR0QsMjAyNCwwLjYyNDA4NjIKQmFyYmFkb3MsQlJCLDIwMTQs
NC45MTMyMzY2CkJhcmJhZG9zLEJSQiwyMDI0LDQuODMyODc0MwpCZWxhcnVzLEJMUiwyMDE0LDYuNzI0NTAxNgpCZWxhcnVzLEJMUiwyMDI0LDYuMTYzMjIx
CkJlbGdpdW0sQkVMLDIwMTQsOC42NjAxMjMKQmVsZ2l1bSxCRUwsMjAyNCw3LjI3OTgzMTQKQmVsaXplLEJMWiwyMDE0LDEuNDM4NTc1OQpCZWxpemUsQkxa
LDIwMjQsMS45MDg1ODUyCkJlbmluLEJFTiwyMDE0LDAuNDMyODIwNwpCZW5pbixCRU4sMjAyNCwwLjQxOTQ0NjYKQmVybXVkYSxCTVUsMjAxNCwxMC40OTMx
MTUKQmVybXVkYSxCTVUsMjAyNCw4LjUwNjA0NwpCaHV0YW4sQlROLDIwMTQsMS44MTcyNzIzCkJodXRhbixCVE4sMjAyNCwyLjA5MzkzOTUKQm9saXZpYSxC
T0wsMjAxNCwxLjgzNDMxODkKQm9saXZpYSxCT0wsMjAyNCwyLjMxMjczMDgKQm9uYWlyZSBTaW50IEV1c3RhdGl1cyBhbmQgU2FiYSxCRVMsMjAxNCw0Ljc4
NzM5MQpCb25haXJlIFNpbnQgRXVzdGF0aXVzIGFuZCBTYWJhLEJFUywyMDI0LDQuOTQ3NDU4CkJvc25pYSBhbmQgSGVyemVnb3ZpbmEsQklILDIwMTQsNS40
MTczMDc0CkJvc25pYSBhbmQgSGVyemVnb3ZpbmEsQklILDIwMjQsNi4xMzkxNzgzCkJvdHN3YW5hLEJXQSwyMDE0LDMuMTUzNzczOApCb3Rzd2FuYSxCV0Es
MjAyNCwyLjk1OTYwNzEKQnJhemlsLEJSQSwyMDE0LDIuNzgxNDQzNgpCcmF6aWwsQlJBLDIwMjQsMi4yNzgzNzE4CkJyaXRpc2ggVmlyZ2luIElzbGFuZHMs
VkdCLDIwMTQsNi41NTMxNzA3CkJyaXRpc2ggVmlyZ2luIElzbGFuZHMsVkdCLDIwMjQsNC44NzM4NDIKQnJ1bmVpLEJSTiwyMDE0LDIxLjE4OTc4NwpCcnVu
ZWksQlJOLDIwMjQsMjYuMDQ2MjAyCkJ1bGdhcmlhLEJHUiwyMDE0LDYuMjUzMTE1CkJ1bGdhcmlhLEJHUiwyMDI0LDQuNjc5Mzg2NgpCdXJraW5hIEZhc28s
QkZBLDIwMTQsMC4xNTczMDEzNwpCdXJraW5hIEZhc28sQkZBLDIwMjQsMC4yNzYzNDgxNwpCdXJ1bmRpLEJESSwyMDE0LDAuMDMyODgyNjgKQnVydW5kaSxC
REksMjAyNCwwLjA2NTI1MzI2NQpDYW1ib2RpYSxLSE0sMjAxNCwwLjQzNTMwODQzCkNhbWJvZGlhLEtITSwyMDI0LDEuMjQwNTk5NApDYW1lcm9vbixDTVIs
MjAxNCwwLjM5Nzc4ODU4CkNhbWVyb29uLENNUiwyMDI0LDAuMzMwNzEyMzUKQ2FuYWRhLENBTiwyMDE0LDE1Ljg1Mzc5MgpDYW5hZGEsQ0FOLDIwMjQsMTMu
NDE5OTE3CkNhcGUgVmVyZGUsQ1BWLDIwMTQsMC45NTgzMTIzMwpDYXBlIFZlcmRlLENQViwyMDI0LDEuMTMxMjgzOQpDZW50cmFsIEFmcmljYW4gUmVwdWJs
aWMsQ0FGLDIwMTQsMC4wMjg1MzA5NjgKQ2VudHJhbCBBZnJpY2FuIFJlcHVibGljLENBRiwyMDI0LDAuMDc0MDM5MzcKQ2hhZCxUQ0QsMjAxNCwwLjE1Mzgz
OTU0CkNoYWQsVENELDIwMjQsMC4xMzk0ODM4NQpDaGlsZSxDSEwsMjAxNCw0LjM0MzExMzQKQ2hpbGUsQ0hMLDIwMjQsMy45ODMxMjQKQ2hpbmEsQ0hOLDIw
MTQsNy4xODc1ODgKQ2hpbmEsQ0hOLDIwMjQsOC42NTgzOQpDb2xvbWJpYSxDT0wsMjAxNCwxLjk3NjYxMDEKQ29sb21iaWEsQ09MLDIwMjQsMS43NTIwNDcK
Q29tb3JvcyxDT00sMjAxNCwwLjIzMTczMTg2CkNvbW9yb3MsQ09NLDIwMjQsMC42MzY1NDQ3CkNvbmdvLENPRywyMDE0LDEuMDk5NTIKQ29uZ28sQ09HLDIw
MjQsMS4zOTU4MzM1CkNvb2sgSXNsYW5kcyxDT0ssMjAxNCw0LjU5MTQ3OQpDb29rIElzbGFuZHMsQ09LLDIwMjQsNS44MjQyMTgzCkNvc3RhIFJpY2EsQ1JJ
LDIwMTQsMS42MTU2MjE5CkNvc3RhIFJpY2EsQ1JJLDIwMjQsMS43MDc3MjA1CkNvdGUgZCdJdm9pcmUsQ0lWLDIwMTQsMC40MDM3NjE4MwpDb3RlIGQnSXZv
aXJlLENJViwyMDI0LDAuNDY0MzQ3NjYKQ3JvYXRpYSxIUlYsMjAxNCw0LjE4Mjk2CkNyb2F0aWEsSFJWLDIwMjQsNC43NjQ3MjMKQ3ViYSxDVUIsMjAxNCwy
LjQyNDg2MDIKQ3ViYSxDVUIsMjAyNCwyLjIyNTkzMQpDdXJhY2FvLENVVywyMDE0LDQxLjE4MjAxCkN1cmFjYW8sQ1VXLDIwMjQsMTIuMzQ0MjA2CkN5cHJ1
cyxDWVAsMjAxNCw1Ljc2NjE0NgpDeXBydXMsQ1lQLDIwMjQsNS4zNzM5MjIKQ3plY2hpYSxDWkUsMjAxNCw5LjkxMDY1NQpDemVjaGlhLENaRSwyMDI0LDcu
MDQzOTY1CkRlbW9jcmF0aWMgUmVwdWJsaWMgb2YgQ29uZ28sQ09ELDIwMTQsMC4wNjQ2MTQxMjUKRGVtb2NyYXRpYyBSZXB1YmxpYyBvZiBDb25nbyxDT0Qs
MjAyNCwwLjA1NDAzMjMzCkRlbm1hcmssRE5LLDIwMTQsNi42NTcxMTM2CkRlbm1hcmssRE5LLDIwMjQsNC43NDYwNTcKRGppYm91dGksREpJLDIwMTQsMC40
MTY0ODUxNgpEamlib3V0aSxESkksMjAyNCwwLjQ4MjY0ODY0CkRvbWluaWNhLERNQSwyMDE0LDIuMzk0NTMzMgpEb21pbmljYSxETUEsMjAyNCwyLjU3OTky
MzQKRG9taW5pY2FuIFJlcHVibGljLERPTSwyMDE0LDIuMDYxMzc5NwpEb21pbmljYW4gUmVwdWJsaWMsRE9NLDIwMjQsMi44OTkyMDM4CkVhc3QgVGltb3Is
VExTLDIwMTQsMC41MzA1NjM2CkVhc3QgVGltb3IsVExTLDIwMjQsMC40NzcwMDU0OApFY3VhZG9yLEVDVSwyMDE0LDIuNzI4OTk3MgpFY3VhZG9yLEVDVSwy
MDI0LDIuNTM1MzM3CkVneXB0LEVHWSwyMDE0LDIuMzMwMTI2NQpFZ3lwdCxFR1ksMjAyNCwyLjIxNzAyMTcKRWwgU2FsdmFkb3IsU0xWLDIwMTQsMC45OTMw
NjI3MwpFbCBTYWx2YWRvcixTTFYsMjAyNCwxLjQxNTU5MDQKRXF1YXRvcmlhbCBHdWluZWEsR05RLDIwMTQsNS40NTg3NDQ1CkVxdWF0b3JpYWwgR3VpbmVh
LEdOUSwyMDI0LDMuNzA0MjM2CkVyaXRyZWEsRVJJLDIwMTQsMC4xODM4Njk1CkVyaXRyZWEsRVJJLDIwMjQsMC4yMTAwNDc3NwpFc3RvbmlhLEVTVCwyMDE0
LDE0LjI5MDI3NQpFc3RvbmlhLEVTVCwyMDI0LDYuMTA1NDY0CkVzd2F0aW5pLFNXWiwyMDE0LDAuNjcwNjkzMTYKRXN3YXRpbmksU1daLDIwMjQsMC44NDA1
MzcyCkV0aGlvcGlhLEVUSCwyMDE0LDAuMTE0MTU4NDgKRXRoaW9waWEsRVRILDIwMjQsMC4xMzUwNjk4NQpFdXJvcGUsT1dJRF9FVVIsMjAxNCw3LjU0NDMw
OTYKRXVyb3BlLE9XSURfRVVSLDIwMjQsNi41NDE1NjEKRXVyb3BlIChleGNsLiBFVS0yNyksLDIwMTQsOC41NTI1NDQKRXVyb3BlIChleGNsLiBFVS0yNyks
LDIwMjQsOC4yOTY4MQpFdXJvcGUgKGV4Y2wuIEVVLTI4KSwsMjAxNCw5LjA0NDYxCkV1cm9wZSAoZXhjbC4gRVUtMjgpLCwyMDI0LDkuNDQ2OTgKRXVyb3Bl
YW4gVW5pb24gKDI3KSxPV0lEX0VVMjcsMjAxNCw2Ljg2MDI3NzcKRXVyb3BlYW4gVW5pb24gKDI3KSxPV0lEX0VVMjcsMjAyNCw1LjM4ODM3MgpFdXJvcGVh
biBVbmlvbiAoMjgpLCwyMDE0LDYuODQ4MDg2NApFdXJvcGVhbiBVbmlvbiAoMjgpLCwyMDI0LDUuMjczNTUzCkZhcm9lIElzbGFuZHMsRlJPLDIwMTQsMTIu
MzEwMjU0CkZhcm9lIElzbGFuZHMsRlJPLDIwMjQsMTMuMDg5MTU3CkZpamksRkpJLDIwMTQsMS4yODU3Njk2CkZpamksRkpJLDIwMjQsMS41NTYxMTExCkZp
bmxhbmQsRklOLDIwMTQsOC43MTkyODYKRmlubGFuZCxGSU4sMjAyNCw1LjMwMDU3OQpGcmFuY2UsRlJBLDIwMTQsNS4wNjcxODQKRnJhbmNlLEZSQSwyMDI0
LDMuOTY5MzY4CkZyZW5jaCBQb2x5bmVzaWEsUFlGLDIwMTQsMy4wNzU1MjcKRnJlbmNoIFBvbHluZXNpYSxQWUYsMjAyNCwzLjI3NDcxNDcKR2Fib24sR0FC
LDIwMTQsMi44NjI2NjE4CkdhYm9uLEdBQiwyMDI0LDIuMTI2Mzg2CkdhbWJpYSxHTUIsMjAxNCwwLjIzMzU5ODQzCkdhbWJpYSxHTUIsMjAyNCwwLjI5MDg2
MDI0Ckdlb3JnaWEsR0VPLDIwMTQsMi4zODgxNjQ1Ckdlb3JnaWEsR0VPLDIwMjQsMy4wOTQ4NzUzCkdlcm1hbnksREVVLDIwMTQsOS43Mzk3NjEKR2VybWFu
eSxERVUsMjAyNCw2Ljc2ODgyMzYKR2hhbmEsR0hBLDIwMTQsMC40NzI3NDY5CkdoYW5hLEdIQSwyMDI0LDAuNjEwNjE0MwpHcmVlY2UsR1JDLDIwMTQsNy4y
MTg4MDgKR3JlZWNlLEdSQywyMDI0LDUuMzEwNjkyCkdyZWVubGFuZCxHUkwsMjAxNCw4Ljk0MjIwMQpHcmVlbmxhbmQsR1JMLDIwMjQsMTEuMDc5ODE5Ckdy
ZW5hZGEsR1JELDIwMTQsMi4xMTg1MTI5CkdyZW5hZGEsR1JELDIwMjQsMy4yMTU5ODIyCkd1YXRlbWFsYSxHVE0sMjAxNCwwLjg1NjA3OTgKR3VhdGVtYWxh
LEdUTSwyMDI0LDEuMDgwMDg0NApHdWluZWEsR0lOLDIwMTQsMC4yMDY2MjYzNwpHdWluZWEsR0lOLDIwMjQsMC4yNzI4NzMyOApHdWluZWEtQmlzc2F1LEdO
QiwyMDE0LDAuMDgyMDcxMDMKR3VpbmVhLUJpc3NhdSxHTkIsMjAyNCwwLjE1NTQ0NDM3Ckd1eWFuYSxHVVksMjAxNCwyLjYxODQyMzIKR3V5YW5hLEdVWSwy
MDI0LDUuNDI2OTkyNApIYWl0aSxIVEksMjAxNCwwLjI2MDEzNDA0CkhhaXRpLEhUSSwyMDI0LDAuMjUyMzQwNApIaWdoLWluY29tZSBjb3VudHJpZXMsT1dJ
RF9ISUMsMjAxNCwxMS4yNTMwNTQKSGlnaC1pbmNvbWUgY291bnRyaWVzLE9XSURfSElDLDIwMjQsOS43OTM4ODMKSG9uZHVyYXMsSE5ELDIwMTQsMS4wNDg1
NTg4CkhvbmR1cmFzLEhORCwyMDI0LDEuMTg3NDM3CkhvbmcgS29uZyxIS0csMjAxNCw2LjIxNTE0MwpIb25nIEtvbmcsSEtHLDIwMjQsNC40OTQxNzQ1Ckh1
bmdhcnksSFVOLDIwMTQsNC40MjQ1NzQKSHVuZ2FyeSxIVU4sMjAyNCw0LjEzNTY2OQpJY2VsYW5kLElTTCwyMDE0LDEwLjUxOTM5NgpJY2VsYW5kLElTTCwy
MDI0LDkuNjY2OTM1CkluZGlhLElORCwyMDE0LDEuNjM2ODg4OQpJbmRpYSxJTkQsMjAyNCwyLjIwMDk3ODMKSW5kb25lc2lhLElETiwyMDE0LDEuOTIzODAx
OApJbmRvbmVzaWEsSUROLDIwMjQsMi44NjUwOTYzCklyYW4sSVJOLDIwMTQsNy45MDQ0MzQ3CklyYW4sSVJOLDIwMjQsOC42NTYyMjcKSXJhcSxJUlEsMjAx
NCwzLjc2MjMxMTUKSXJhcSxJUlEsMjAyNCw1LjA3NDM5OApJcmVsYW5kLElSTCwyMDE0LDcuOTEwNjkzCklyZWxhbmQsSVJMLDIwMjQsNi4zMzg4MjQzCklz
cmFlbCxJU1IsMjAxNCw3LjYzMjk2MTMKSXNyYWVsLElTUiwyMDI0LDUuNjA3NDQxNApJdGFseSxJVEEsMjAxNCw1Ljc1OTM5NTYKSXRhbHksSVRBLDIwMjQs
NS4wODc4ODYKSmFtYWljYSxKQU0sMjAxNCwyLjc0NzM2NgpKYW1haWNhLEpBTSwyMDI0LDIuOTU5MjE5NwpKYXBhbixKUE4sMjAxNCw5Ljg4NTUyOQpKYXBh
bixKUE4sMjAyNCw3Ljc3MjQ3NDMKSm9yZGFuLEpPUiwyMDE0LDIuOTc2MDgKSm9yZGFuLEpPUiwyMDI0LDIuMDA5ODEzCkthemFraHN0YW4sS0FaLDIwMTQs
MTYuNzkwMTA4CkthemFraHN0YW4sS0FaLDIwMjQsMTMuOTM3MzUyCktlbnlhLEtFTiwyMDE0LDAuMjg2OTQ4OApLZW55YSxLRU4sMjAyNCwwLjM3NjExNzc3
CktpcmliYXRpLEtJUiwyMDE0LDAuNDc2NTA4MzgKS2lyaWJhdGksS0lSLDIwMjQsMC41Mzg2MjIyCktvc292byxPV0lEX0tPUywyMDE0LDMuOTQwOTQ3NQpL
b3Nvdm8sT1dJRF9LT1MsMjAyNCw0Ljc3Nzk5MQpLdXdhaXQsS1dULDIwMTQsMjAuNDU3ODQ2Ckt1d2FpdCxLV1QsMjAyNCwyNi4yNDc1MwpLeXJneXpzdGFu
LEtHWiwyMDE0LDEuNzM1NDc0NgpLeXJneXpzdGFuLEtHWiwyMDI0LDEuNjM3OTI1OQpMYW9zLExBTywyMDE0LDAuNjUzODcyMjUKTGFvcyxMQU8sMjAyNCwz
LjE0MDM0NQpMYXR2aWEsTFZBLDIwMTQsMy41OTY2MTQ2CkxhdHZpYSxMVkEsMjAyNCwzLjQ1MjA5NjIKTGViYW5vbixMQk4sMjAxNCwzLjc4MDA2MjIKTGVi
YW5vbixMQk4sMjAyNCwyLjY5NTUwMDQKTGVzb3RobyxMU08sMjAxNCwxLjA3ODAxNDQKTGVzb3RobyxMU08sMjAyNCwxLjEwMDI1NQpMaWJlcmlhLExCUiwy
MDE0LDAuMTYwNzA5NDYKTGliZXJpYSxMQlIsMjAyNCwwLjE1MzEwODQzCkxpYnlhLExCWSwyMDE0LDEwLjYzNDE0MwpMaWJ5YSxMQlksMjAyNCw4Ljg0MTU5
NgpMaWVjaHRlbnN0ZWluLExJRSwyMDE0LDQuMzM0MDA1NApMaWVjaHRlbnN0ZWluLExJRSwyMDI0LDMuMjk5MjYwNgpMaXRodWFuaWEsTFRVLDIwMTQsNC4z
Nzk0OTMKTGl0aHVhbmlhLExUVSwyMDI0LDQuMzg2NTA1Ckxvdy1pbmNvbWUgY291bnRyaWVzLE9XSURfTElDLDIwMTQsMC4zMDM2NDAzNApMb3ctaW5jb21l
IGNvdW50cmllcyxPV0lEX0xJQywyMDI0LDAuMjc4NjU3MQpMb3dlci1taWRkbGUtaW5jb21lIGNvdW50cmllcyxPV0lEX0xNQywyMDE0LDEuMzE3OTU0Ckxv
d2VyLW1pZGRsZS1pbmNvbWUgY291bnRyaWVzLE9XSURfTE1DLDIwMjQsMS41ODgyNjM1Ckx1eGVtYm91cmcsTFVYLDIwMTQsMTcuNjI5MTkKTHV4ZW1ib3Vy
ZyxMVVgsMjAyNCwxMC40NTk2NDkKTWFjYW8sTUFDLDIwMTQsMi4xMTkyNDQ4Ck1hY2FvLE1BQywyMDI0LDEuNDY2NTcyMwpNYWRhZ2FzY2FyLE1ERywyMDE0
LDAuMTE5OTcwOTgKTWFkYWdhc2NhcixNREcsMjAyNCwwLjE0MTY2NzA0Ck1hbGF3aSxNV0ksMjAxNCwwLjA2MTA3MjA0Ck1hbGF3aSxNV0ksMjAyNCwwLjA4
Njg4MzEyCk1hbGF5c2lhLE1ZUywyMDE0LDcuOTgzMgpNYWxheXNpYSxNWVMsMjAyNCw4LjE2MjA2Ck1hbGRpdmVzLE1EViwyMDE0LDMuMTg4OTkwOApNYWxk
aXZlcyxNRFYsMjAyNCw0LjM2OTQ2ODcKTWFsaSxNTEksMjAxNCwwLjE3NDIxMDEzCk1hbGksTUxJLDIwMjQsMC4yODUwOTk3Ck1hbHRhLE1MVCwyMDE0LDUu
NTYzNTI5Ck1hbHRhLE1MVCwyMDI0LDMuMjAzNzIwNgpNYXJzaGFsbCBJc2xhbmRzLE1ITCwyMDE0LDIuNzk0OTgxNQpNYXJzaGFsbCBJc2xhbmRzLE1ITCwy
MDI0LDQuMTExNTcyMwpNYXVyaXRhbmlhLE1SVCwyMDE0LDAuNjY1OTA1ODMKTWF1cml0YW5pYSxNUlQsMjAyNCwxLjAxMzQwMTIKTWF1cml0aXVzLE1VUywy
MDE0LDMuMjU1NjEzMwpNYXVyaXRpdXMsTVVTLDIwMjQsMy42NzY2MjI2Ck1leGljbyxNRVgsMjAxNCw0LjA0MDk2OQpNZXhpY28sTUVYLDIwMjQsMy41MjI3
Mjc1Ck1pY3JvbmVzaWEgKGNvdW50cnkpLEZTTSwyMDE0LDEuMjg0MzQ1MQpNaWNyb25lc2lhIChjb3VudHJ5KSxGU00sMjAyNCwxLjMzMDk1OTMKTW9sZG92
YSxNREEsMjAxNCwxLjQyMjU0MTkKTW9sZG92YSxNREEsMjAyNCwxLjc1NTE1NjgKTW9uZ29saWEsTU5HLDIwMTQsMTAuMTcxNDI4Ck1vbmdvbGlhLE1ORywy
MDI0LDEyLjg1OTU0OQpNb250ZW5lZ3JvLE1ORSwyMDE0LDMuMzM2ODMzNwpNb250ZW5lZ3JvLE1ORSwyMDI0LDMuNzIxNzM2NApNb250c2VycmF0LE1TUiwy
MDE0LDkuMjc5ODY1Ck1vbnRzZXJyYXQsTVNSLDIwMjQsNi4wMjE1MjI1Ck1vcm9jY28sTUFSLDIwMTQsMS42NzE1NDczCk1vcm9jY28sTUFSLDIwMjQsMS44
MTM1NTYKTW96YW1iaXF1ZSxNT1osMjAxNCwwLjMxNDgwODM0Ck1vemFtYmlxdWUsTU9aLDIwMjQsMC4yNDg2OTE2MgpNeWFubWFyLE1NUiwyMDE0LDAuMzEy
NzIzNjcKTXlhbm1hcixNTVIsMjAyNCwwLjU3OTg3ODcKTmFtaWJpYSxOQU0sMjAxNCwxLjM0MDM3NTIKTmFtaWJpYSxOQU0sMjAyNCwxLjE0NDg3MzUKTmF1
cnUsTlJVLDIwMTQsNC43NjQ2Mjk0Ck5hdXJ1LE5SVSwyMDI0LDUuMTI5NTEyCk5lcGFsLE5QTCwyMDE0LDAuMjc0MTk2NTcKTmVwYWwsTlBMLDIwMjQsMC42
MzMyMzQ5Ck5ldGhlcmxhbmRzLE5MRCwyMDE0LDkuMjQ4NTA1Ck5ldGhlcmxhbmRzLE5MRCwyMDI0LDYuMjk2OTEKTmV3IENhbGVkb25pYSxOQ0wsMjAxNCwx
Ny44MTEwOTYKTmV3IENhbGVkb25pYSxOQ0wsMjAyNCwxOC4wNjQ0Ck5ldyBaZWFsYW5kLE5aTCwyMDE0LDcuODI2NTU4NgpOZXcgWmVhbGFuZCxOWkwsMjAy
NCw2LjIyOTM0MwpOaWNhcmFndWEsTklDLDIwMTQsMC43NzgwMDkzCk5pY2FyYWd1YSxOSUMsMjAyNCwwLjgxNDA4ODQKTmlnZXIsTkVSLDIwMTQsMC4xMTc0
MTYyCk5pZ2VyLE5FUiwyMDI0LDAuMTE3MjY1MDYKTmlnZXJpYSxOR0EsMjAxNCwwLjY2Njg4MTYKTmlnZXJpYSxOR0EsMjAyNCwwLjU4MzczODYKTml1ZSxO
SVUsMjAxNCw0LjA4OTI4NgpOaXVlLE5JVSwyMDI0LDQuMTQyODU3Ck5vcnRoIEFtZXJpY2EsT1dJRF9OQU0sMjAxNCwxMi4wMDgxNTcKTm9ydGggQW1lcmlj
YSxPV0lEX05BTSwyMDI0LDkuOTkwNzE3Ck5vcnRoIEFtZXJpY2EgKGV4Y2wuIFVTQSksLDIwMTQsNS4xMTAwODMKTm9ydGggQW1lcmljYSAoZXhjbC4gVVNB
KSwsMjAyNCw0LjQ3NzY0NDQKTm9ydGggS29yZWEsUFJLLDIwMTQsMS41NjQ3Mzc4Ck5vcnRoIEtvcmVhLFBSSywyMDI0LDIuMzYwNDkxOApOb3J0aCBNYWNl
ZG9uaWEsTUtELDIwMTQsMy41ODcyNjI2Ck5vcnRoIE1hY2Vkb25pYSxNS0QsMjAyNCwzLjYzMTE1NTMKTm9yd2F5LE5PUiwyMDE0LDguNzU0NDM0Ck5vcndh
eSxOT1IsMjAyNCw2LjY2NzYxNgpPY2VhbmlhLE9XSURfT0NFLDIwMTQsMTEuMTY2MjA0Ck9jZWFuaWEsT1dJRF9PQ0UsMjAyNCw5LjUzMzQ3MQpPbWFuLE9N
TiwyMDE0LDE2LjUzNzQwNQpPbWFuLE9NTiwyMDI0LDE1LjY1MTEwNwpQYWtpc3RhbixQQUssMjAxNCwwLjcwMzg0NjkKUGFraXN0YW4sUEFLLDIwMjQsMC43
MTU1Njg4NApQYWxhdSxQTFcsMjAxNCwxMi4zOTIzMzQKUGFsYXUsUExXLDIwMjQsMTIuNzYwMjU4ClBhbGVzdGluZSxQU0UsMjAxNCwwLjY0MjIxMjEKUGFs
ZXN0aW5lLFBTRSwyMDI0LDAuODY5ODE5MzQKUGFuYW1hLFBBTiwyMDE0LDIuNzQ2MjM1ClBhbmFtYSxQQU4sMjAyNCwyLjgwNDYwOTUKUGFwdWEgTmV3IEd1
aW5lYSxQTkcsMjAxNCwwLjcxODU5MzY2ClBhcHVhIE5ldyBHdWluZWEsUE5HLDIwMjQsMC43ODkwNTQ1ClBhcmFndWF5LFBSWSwyMDE0LDAuODk4MDg1NgpQ
YXJhZ3VheSxQUlksMjAyNCwxLjE0NTQwOTcKUGVydSxQRVIsMjAxNCwxLjYzOTkzNTkKUGVydSxQRVIsMjAyNCwyLjA1MjkyNDIKUGhpbGlwcGluZXMsUEhM
LDIwMTQsMC45NjQ0MDc3ClBoaWxpcHBpbmVzLFBITCwyMDI0LDEuNTA5NjAzNQpQb2xhbmQsUE9MLDIwMTQsOC4wODYwOTYKUG9sYW5kLFBPTCwyMDI0LDcu
MDgwMTA5NgpQb3J0dWdhbCxQUlQsMjAxNCw0LjYwNzA5MgpQb3J0dWdhbCxQUlQsMjAyNCwzLjQwODkwNzQKUWF0YXIsUUFULDIwMTQsNDEuMzEwMgpRYXRh
cixRQVQsMjAyNCw0MS4yNzExOApSb21hbmlhLFJPVSwyMDE0LDMuOTczMDg5ClJvbWFuaWEsUk9VLDIwMjQsMy42MDUxMDQKUnVzc2lhLFJVUywyMDE0LDEx
LjI5NzU5NQpSdXNzaWEsUlVTLDIwMjQsMTIuMjk0NzA1ClJ3YW5kYSxSV0EsMjAxNCwwLjA3MzAxNTU1ClJ3YW5kYSxSV0EsMjAyNCwwLjE0MjYzOTQ3ClNh
aW50IEhlbGVuYSxTSE4sMjAxNCwyLjAwMTA5MjIKU2FpbnQgSGVsZW5hLFNITiwyMDI0LDIuMTY0NzY2MwpTYWludCBLaXR0cyBhbmQgTmV2aXMsS05BLDIw
MTQsNC44MjE4NzEzClNhaW50IEtpdHRzIGFuZCBOZXZpcyxLTkEsMjAyNCw1LjU0MjQyOQpTYWludCBMdWNpYSxMQ0EsMjAxNCwyLjc5ODMxMTcKU2FpbnQg
THVjaWEsTENBLDIwMjQsMi45ODY5NTk3ClNhaW50IFBpZXJyZSBhbmQgTWlxdWVsb24sU1BNLDIwMTQsMTEuMDIxMzkKU2FpbnQgUGllcnJlIGFuZCBNaXF1
ZWxvbixTUE0sMjAyNCw5Ljc4ODAzOQpTYWludCBWaW5jZW50IGFuZCB0aGUgR3JlbmFkaW5lcyxWQ1QsMjAxNCwyLjM4NDY0MDUKU2FpbnQgVmluY2VudCBh
bmQgdGhlIEdyZW5hZGluZXMsVkNULDIwMjQsMi41NDA1NjQzClNhbW9hLFdTTSwyMDE0LDEuMDA3MzQzMgpTYW1vYSxXU00sMjAyNCwxLjEyNjM4NTcKU2Fv
IFRvbWUgYW5kIFByaW5jaXBlLFNUUCwyMDE0LDAuNjUxNjkyMwpTYW8gVG9tZSBhbmQgUHJpbmNpcGUsU1RQLDIwMjQsMC42MDE5MTEzClNhdWRpIEFyYWJp
YSxTQVUsMjAxNCwyMC4xNzc0NzkKU2F1ZGkgQXJhYmlhLFNBVSwyMDI0LDIwLjM3OTE5NApTZW5lZ2FsLFNFTiwyMDE0LDAuNjE2NDcwOTMKU2VuZWdhbCxT
RU4sMjAyNCwwLjc2MTYzOTU0ClNlcmJpYSxTUkIsMjAxNCw1LjE2NTM1MQpTZXJiaWEsU1JCLDIwMjQsNi4yMzc4MzkKU2V5Y2hlbGxlcyxTWUMsMjAxNCw0
LjM3NTk0OTQKU2V5Y2hlbGxlcyxTWUMsMjAyNCw1LjAwMjk5OApTaWVycmEgTGVvbmUsU0xFLDIwMTQsMC4xNjQxMTQxMwpTaWVycmEgTGVvbmUsU0xFLDIw
MjQsMC4xNjU5NjYKU2luZ2Fwb3JlLFNHUCwyMDE0LDguNzIxMzA1ClNpbmdhcG9yZSxTR1AsMjAyNCw5LjI0NDgzNgpTaW50IE1hYXJ0ZW4gKER1dGNoIHBh
cnQpLFNYTSwyMDE0LDE5Ljc3MDMyMwpTaW50IE1hYXJ0ZW4gKER1dGNoIHBhcnQpLFNYTSwyMDI0LDE2LjU0NjI3NApTbG92YWtpYSxTVkssMjAxNCw2LjIy
NDY4OQpTbG92YWtpYSxTVkssMjAyNCw1LjI4Mjc3MwpTbG92ZW5pYSxTVk4sMjAxNCw2LjU5MTcwODcKU2xvdmVuaWEsU1ZOLDIwMjQsNi4wMTg0Mjc0ClNv
bG9tb24gSXNsYW5kcyxTTEIsMjAxNCwwLjUzNDUzMDcKU29sb21vbiBJc2xhbmRzLFNMQiwyMDI0LDAuMzU5Njc1MQpTb21hbGlhLFNPTSwyMDE0LDAuMDY2
NDcwODcKU29tYWxpYSxTT00sMjAyNCwwLjA2OTM5NzM1ClNvdXRoIEFmcmljYSxaQUYsMjAxNCw4LjY2MzM1NgpTb3V0aCBBZnJpY2EsWkFGLDIwMjQsNi44
NzE1Nzk2ClNvdXRoIEFtZXJpY2EsT1dJRF9TQU0sMjAxNCwzLjAxMDg0MzgKU291dGggQW1lcmljYSxPV0lEX1NBTSwyMDI0LDIuNTQ5NjA0NwpTb3V0aCBL
b3JlYSxLT1IsMjAxNCwxMi40NTMwNzIKU291dGggS29yZWEsS09SLDIwMjQsMTEuMjg1ODkzClNvdXRoIFN1ZGFuLFNTRCwyMDE0LDAuMTMzMzgzNzcKU291
dGggU3VkYW4sU1NELDIwMjQsMC4xNDE5NjU0OApTcGFpbixFU1AsMjAxNCw1LjQyOTI5ODQKU3BhaW4sRVNQLDIwMjQsNC41OTkwMTUKU3JpIExhbmthLExL
QSwyMDE0LDAuNzkyNDg0MTYKU3JpIExhbmthLExLQSwyMDI0LDAuOTAxMzk5NzMKU3VkYW4sU0ROLDIwMTQsMC4zMzcyOTM1NApTdWRhbixTRE4sMjAyNCww
LjM1MjcyODc4ClN1cmluYW1lLFNVUiwyMDE0LDUuNTYzMTUxNApTdXJpbmFtZSxTVVIsMjAyNCw0LjcwOTMxNDMKU3dlZGVuLFNXRSwyMDE0LDQuNDc0Mjk3
NQpTd2VkZW4sU1dFLDIwMjQsMy41OTE2NTQzClN3aXR6ZXJsYW5kLENIRSwyMDE0LDQuNzkxNTc2ClN3aXR6ZXJsYW5kLENIRSwyMDI0LDMuNTk0Njg1NgpT
eXJpYSxTWVIsMjAxNCwxLjcxOTkzMTcKU3lyaWEsU1lSLDIwMjQsMS4yODc5OTc0ClRhaXdhbixUV04sMjAxNCwxMS43OTkxNDEKVGFpd2FuLFRXTiwyMDI0
LDExLjMwMTE2NApUYWppa2lzdGFuLFRKSywyMDE0LDAuNTQ2Njk2MwpUYWppa2lzdGFuLFRKSywyMDI0LDEuMDE0MDUzMwpUYW56YW5pYSxUWkEsMjAxNCww
LjIxODQxMzIKVGFuemFuaWEsVFpBLDIwMjQsMC4yOTE5ODI0NApUaGFpbGFuZCxUSEEsMjAxNCwzLjg4MDY4NgpUaGFpbGFuZCxUSEEsMjAyNCwzLjczNjEx
ODgKVG9nbyxUR08sMjAxNCwwLjIxNDg1NwpUb2dvLFRHTywyMDI0LDAuMzI3NTUyMTcKVG9uZ2EsVE9OLDIwMTQsMC45OTY1MjA2ClRvbmdhLFRPTiwyMDI0
LDEuNDY0MDI4ClRyaW5pZGFkIGFuZCBUb2JhZ28sVFRPLDIwMTQsMzIuOTc3NzM3ClRyaW5pZGFkIGFuZCBUb2JhZ28sVFRPLDIwMjQsMjIuOTMxOTQ0ClR1
bmlzaWEsVFVOLDIwMTQsMi42NDk3OTUzClR1bmlzaWEsVFVOLDIwMjQsMi42NjA1Mzc3ClR1cmtleSxUVVIsMjAxNCw0LjcwNjQyOTUKVHVya2V5LFRVUiwy
MDI0LDUuODY1MDA5ClR1cmttZW5pc3RhbixUS00sMjAxNCwxMC4zMzY4NTQKVHVya21lbmlzdGFuLFRLTSwyMDI0LDEwLjgwODY3NQpUdXJrcyBhbmQgQ2Fp
Y29zIElzbGFuZHMsVENBLDIwMTQsOC43NzQyOTcKVHVya3MgYW5kIENhaWNvcyBJc2xhbmRzLFRDQSwyMDI0LDguMTM2MTE3ClR1dmFsdSxUVVYsMjAxNCww
LjY2NTc1ODEzClR1dmFsdSxUVVYsMjAyNCwxLjE4MzE2MjcKVWdhbmRhLFVHQSwyMDE0LDAuMTEyMzY3MjIKVWdhbmRhLFVHQSwyMDI0LDAuMTI2NjM3OTgK
VWtyYWluZSxVS1IsMjAxNCw1LjYwNTYwNgpVa3JhaW5lLFVLUiwyMDI0LDMuNzYzNzg4MgpVbml0ZWQgQXJhYiBFbWlyYXRlcyxBUkUsMjAxNCwyNi4wNDUw
OTcKVW5pdGVkIEFyYWIgRW1pcmF0ZXMsQVJFLDIwMjQsMjAuMTMxMDc1ClVuaXRlZCBLaW5nZG9tLEdCUiwyMDE0LDYuNzY0ODMzClVuaXRlZCBLaW5nZG9t
LEdCUiwyMDI0LDQuNTI1Nzk5MwpVbml0ZWQgU3RhdGVzLFVTQSwyMDE0LDE3LjExODkxNwpVbml0ZWQgU3RhdGVzLFVTQSwyMDI0LDE0LjE5NzI4NwpVcHBl
ci1taWRkbGUtaW5jb21lIGNvdW50cmllcyxPV0lEX1VNQywyMDE0LDUuMzQxNTUxMwpVcHBlci1taWRkbGUtaW5jb21lIGNvdW50cmllcyxPV0lEX1VNQywy
MDI0LDYuMDY4ODQ0MwpVcnVndWF5LFVSWSwyMDE0LDEuOTkxMzk5MwpVcnVndWF5LFVSWSwyMDI0LDIuMzU0MzcKVXpiZWtpc3RhbixVWkIsMjAxNCwzLjU1
Njc0OTYKVXpiZWtpc3RhbixVWkIsMjAyNCwzLjgyOTEyNwpWYW51YXR1LFZVVCwyMDE0LDAuNjA1Mzk0MQpWYW51YXR1LFZVVCwyMDI0LDAuNjAxMjQyOQpW
ZW5lenVlbGEsVkVOLDIwMTQsNS44MTcxMTk2ClZlbmV6dWVsYSxWRU4sMjAyNCw0LjA4NTIwMwpWaWV0bmFtLFZOTSwyMDE0LDEuOTYzODc4ClZpZXRuYW0s
Vk5NLDIwMjQsMy42NzMwMzUxCldhbGxpcyBhbmQgRnV0dW5hLFdMRiwyMDE0LDIuMDgwNDY3MgpXYWxsaXMgYW5kIEZ1dHVuYSxXTEYsMjAyNCwyLjY5OTEw
NjUKV29ybGQsT1dJRF9XUkwsMjAxNCw0LjgwNDU5NzQKV29ybGQsT1dJRF9XUkwsMjAyNCw0LjcyOTA3NQpZZW1lbixZRU0sMjAxNCwwLjg3OTc3MDE2Clll
bWVuLFlFTSwyMDI0LDAuMjQ5NDQwNTgKWmFtYmlhLFpNQiwyMDE0LDAuMzMyMzc4NApaYW1iaWEsWk1CLDIwMjQsMC41NjUwMTYyClppbWJhYndlLFpXRSwy
MDE0LDAuODQwODQxNzcKWmltYmFid2UsWldFLDIwMjQsMC44MjM2NjU1Ngo="""
)
_DATA["percapita_co2_by_source"] = (
"""RW50aXR5LENvZGUsWWVhcixDb2FsLE9pbCxHYXMsRmxhcmluZyxDZW1lbnQsT3RoZXIgaW5kdXN0cnkKQXVzdHJhbGlhLEFVUywyMDI0LDUuMzM2MjgzLDUu
NDU3NTU2NywyLjgyMDgwNTMsMC42MDg2NDgzLDAuMTAyMzA2NDgsMC4xNTE1OTg2OQpCZWxnaXVtLEJFTCwyMDI0LDAuOTc4OTkyNzYsMy41NzQ0NTQ1LDIu
MzkxNjU4OCwwLjAwNjMxNDI5NSwwLjE4NTc2MDIzLDAuMTQyNjUxCkNoaW5hLENITiwyMDI0LDYuMjYwNzUzLDEuMjAxNTUwMSwwLjYyNjY0MjA1LDAuMDAy
ODI3NDk0NCwwLjQzNTI0NDU2LDAuMTMxMzcxOTUKR2VybWFueSxERVUsMjAyNCwxLjkyMzUzNzUsMi43NTUxOTI4LDEuODY0ODY4MywwLjAxOTUyMjM4Miww
LjEyMTYxMjA4LDAuMDg0MDkwNDIKSW5kaWEsSU5ELDIwMjQsMS40NTI4NzgxLDAuNTE1MDY5NTQsMC4xMDI5MzUwNiwwLjAwMTkyMzI2OTEsMC4xMjgxNzIy
MiwKSmFwYW4sSlBOLDIwMjQsMy4wNzg0ODk1LDIuODU3MjQzMywxLjU3ODEzMjMsMC4wMDI3MTIzNjk3LDAuMTY0MDcyMjYsMC4wOTE4MjQyMgpRYXRhcixR
QVQsMjAyNCwwLjAwMTM5NTgwMSw2LjM1MjIxMzQsMzMuNzA5MjY3LDAuNTcyMzQ0OTYsMC42MzU5NTY3NiwKU2luZ2Fwb3JlLFNHUCwyMDI0LDAuMTk1ODkx
MzIsNS40NTYwMzc1LDMuNTkyOTA3LDAsMCwKU291dGggQWZyaWNhLFpBRiwyMDI0LDUuODAxODE5MywwLjg2MDY0MzgsMC4xMjEwMTEyMDUsMC4wMDAwNzc1
MzgxNywwLjA4ODAyNzc0LApUcmluaWRhZCBhbmQgVG9iYWdvLFRUTywyMDI0LCwyLjg3MzcwMDksMTkuNzcyNjMsMC4xMTgzNDAzNjYsMC4xNjcyNjkzMywK
VW5pdGVkIEFyYWIgRW1pcmF0ZXMsQVJFLDIwMjQsMC4zOTE0NzMyLDUuOTQ1MjE2NywxMy4wOTk3NzgsMC4xMzc4NzkxNSwwLjU1NjcyODUsClVuaXRlZCBL
aW5nZG9tLEdCUiwyMDI0LDAuMjQ1MTMyMzYsMi4zMDg0MjEsMS44MzE0MDUsMC4wNDk0NzE1NDYsMC4wNTI0MTg1ODYsMC4wMzg5NTA4NjQKVW5pdGVkIFN0
YXRlcyxVU0EsMjAyNCwyLjEyMTY4NDgsNi4zMzc1NTgzLDUuMDYwODA4NywwLjE3NzU0MzI3LDAuMTA3ODk3NDgsMC4zOTE3OTM1MgpXb3JsZCxPV0lEX1dS
TCwyMDI0LDEuOTM2NDUwMiwxLjUyNzg5LDAuOTgxMzU5MywwLjA1MDkzMjI1LDAuMTgwNDQ4NzMsMC4wNTE5OTQyMzg="""
)
_DATA["us_co2_by_fuel"] = (
"""RW50aXR5LENvZGUsWWVhcixPaWwsQ29hbCxDZW1lbnQsR2FzLEZsYXJpbmcsT3RoZXIgaW5kdXN0cnkKVW5pdGVkIFN0YXRlcyxVU0EsMTgwMCwwLDI1Mjgx
NS45OCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODAxLDAsMjY3NDcyLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4MDIsMCwyODk0NTYsMCwwLCwKVW5p
dGVkIFN0YXRlcyxVU0EsMTgwMywwLDI5Njc4NCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODA0LDAsMzMzNDI0LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNB
LDE4MDUsMCwzNDA3NTIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgwNiwwLDMzMzQyNCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODA3LDAsMzc3Mzky
LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4MDgsMCwzOTIwNDgsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgwOSwwLDQwMzA0MCwwLDAsLApVbml0ZWQg
U3RhdGVzLFVTQSwxODEwLDAsNDE3Njk2LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4MTEsMCw0NDcwMDgsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgx
MiwwLDQ4MzY0OCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODEzLDAsNTIwMjg4LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4MTQsMCw1NjA1OTIsMCww
LCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgxNSwwLDYwMDg5NiwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODE2LDAsNjYzMTg0LDAsMCwsClVuaXRlZCBTdGF0
ZXMsVVNBLDE4MTcsMCw3MTgxNDQsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgxOCwwLDc4MDQzMiwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODE5LDAs
NzYyMTEyLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4MjAsMCw3OTE0MjQsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgyMSwwLDgyODA2NCwwLDAsLApV
bml0ZWQgU3RhdGVzLFVTQSwxODIyLDAsODY0NzA0LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4MjMsMCw5MDEzNDQsMCwwLCwKVW5pdGVkIFN0YXRlcyxV
U0EsMTgyNCwwLDEwMTQ5MjgsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgyNSwwLDExMzU4NDAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgyNiwwLDEz
MTUzNzYsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgyNywwLDE0NDcyODAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgyOCwwLDE1OTM4NDAsMCwwLCwK
VW5pdGVkIFN0YXRlcyxVU0EsMTgyOSwwLDE3OTUzNjAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzMCwwLDIwODg0ODAsMCwwLCwKVW5pdGVkIFN0YXRl
cyxVU0EsMTgzMSwwLDIyNjQzNTIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzMiwwLDMwMjI4MDAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzMyww
LDM1Mjg0MzIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzNCwwLDMzODE4NzIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzNSwwLDQzMTYxOTIsMCww
LCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzNiwwLDQ3MzAyMjQsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzNywwLDUzMDU0NzIsMCwwLCwKVW5pdGVkIFN0
YXRlcyxVU0EsMTgzOCwwLDUwMzA2NzIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTgzOSwwLDU1MTc5ODQsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg0
MCwwLDU4NzMzOTIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg0MSwwLDYyMTQxNDQsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg0MiwwLDY5MTc2MzIs
MCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg0MywwLDc3NjQwMTYsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg0NCwwLDkzMDY1NjAsMCwwLCwKVW5pdGVk
IFN0YXRlcyxVU0EsMTg0NSwwLDExMjA0NTEyLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NDYsMCwxMjcxMDQxNiwwLDAsLApVbml0ZWQgU3RhdGVzLFVT
QSwxODQ3LDAsMTUwNzAwMzIsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg0OCwwLDE2Nzg0Nzg0LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NDksMCwx
ODIyMTA3MiwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODUwLDAsMTk3OTI5MjgsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg1MSwwLDI0NjMzMDcyLDAs
MCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NTIsMCwyNjc5MTE2OCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODUzLDAsMzAxNjIwNDgsMCwwLCwKVW5pdGVk
IFN0YXRlcyxVU0EsMTg1NCwwLDMzMTU5MTk4LDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NTUsMCwzODE2MDU2MCwwLDAsLApVbml0ZWQgU3RhdGVzLFVT
QSwxODU2LDAsNDAwMzY1MzAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg1NywwLDQxMDU1MTIwLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NTgsMCw0
MTY0ODY5MCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODU5LDAsNDUzMjAwMTYsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg2MCwyMDE1MzYsNDcyMzYy
NzAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg2MSw4NjEwNDAsNDQ4MTgwNTAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg2MiwxMjQ5MzI4LDQ2MTk5
NDcwLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NjMsMTA2NjIyNCw1Mzc0MzU1MCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODY0LDg2NDcwNCw1Nzc5
MjI3MCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODY1LDEwMjIyNTYsNTc3OTk2MDAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg2NiwxNDY5MjY0LDU3
NzYyOTYwLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NjcsMTM2Njc0MSw3MTQ5OTIzMCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODY4LDE0OTEyNDgs
ODA4NjA4MjAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg2OSwxNzI1NzQ0LDkxOTcwMDYwLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NzAsMjA0ODE3
Niw5NjU3MjA1MCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODcxLDIwMTg4NjMuOSwxMDEwMDE4MjAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg3Miwy
NDEwOTEyLDEyMzg5NDUwMCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODczLDM4NTA4NjQsMTM1NjU1OTQwLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4
NzQsNDMyMzUyMCwxMjk5NzMwNzAsMCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg3NSwzNDI5NTA0LDEzMjMxODA0MCwwLDAsLApVbml0ZWQgU3RhdGVzLFVT
QSwxODc2LDM0ODQzNjgsMTI5MzM1NjMwLDAsMCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4NzcsNTE3NzEwMywxNDE5OTg0NTAsMCwwLCwKVW5pdGVkIFN0YXRl
cyxVU0EsMTg3OCw2MDU2NTkyLDEzNzg0NzAwMCwwLDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODc5LDc4NjI5NDQsMTY3NjA2MDIwLDAsMCwsClVuaXRlZCBT
dGF0ZXMsVVNBLDE4ODAsMTAzODM3NzYsMTg4MzAwMjkwLDE3MjMyNCwwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg4MSwxMDkxNTA1NiwxOTkyNzc2MzAsMjA3
ODI1LDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxODgyLDExOTY2NjI0LDIyMzI2MjE4MCwyNzAxNzMsMTY0ODgwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg4Myw5
MDA1OTgyLDI0NDQ5NTIwMCwzNDgzMTUsMzgxMDUxLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg4NCw5MTE2MDMyLDI1NzMwMDc1MCwzMzI1MjAsMTE3MjQ4MCws
ClVuaXRlZCBTdGF0ZXMsVVNBLDE4ODUsODEzNDA4MCwyNjAzMTI1MzAsMzQ0OTkwLDM3MTUyOTYsLApVbml0ZWQgU3RhdGVzLFVTQSwxODg2LDEwNzI0NTI4
LDI2OTA2MjE4MCwzNzQwODUsNzY3NjA4MCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4ODcsMTA3Njg0OTYsMjg0ODI0NzAwLDU1NjM2OCwxMTc4NzA4OCwsClVu
aXRlZCBTdGF0ZXMsVVNBLDE4ODgsMTA1MjY2NzIsMzQ2NTc0MTAwLDU0MDYxOSwxNjc3Mzc5MiwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4ODksMTM1Mzg0ODAs
MzA5Nzc2NTQwLDU4MTkxMCwxMjIyNjc2OCwsClVuaXRlZCBTdGF0ZXMsVVNBLDE4OTAsMTc3ODg1NTgsMzcyMjY5OTgwLDY2NTA0MSwxMTY4ODA1MywsClVu
aXRlZCBTdGF0ZXMsVVNBLDE4OTEsMjEyNTEyMDAsMzk3MjI1MjAwLDY4MzU2MSw4OTQ3NDg4LCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg5MiwxOTYyODA0OCw0
MjMwNDU0NDAsNzI4MTA1LDc3NzUwMDgsLApVbml0ZWQgU3RhdGVzLFVTQSwxODkzLDE4NzAxMDU2LDQyODEwNTQ0MCw2NjUyNDYsNzI4NzY5NiwsClVuaXRl
ZCBTdGF0ZXMsVVNBLDE4OTQsMTg5NzYwMjAsMzk5Mzc1ODAwLDY5NTE1NCw2Njk3ODUwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg5NSwyMDUzMjkwMCw0NTIz
NzIzMjAsNzI1ODQyLDcwNDIxNTQsLApVbml0ZWQgU3RhdGVzLFVTQSwxODk2LDIzODM3ODAyLDQ1MDE5NTk0MCw3OTA4NTYsNjg0Nzk2NCwsClVuaXRlZCBT
dGF0ZXMsVVNBLDE4OTcsMjM1MzM3MDAsNDY5NDY0ODYwLDkxMzU1NSw3Mjg3NjQzLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg5OCwyMTUwNzY4MCw1MTUzNTI2
MDAsMTAwNjgwNC45NCw4NDYwMTc2LCwKVW5pdGVkIFN0YXRlcyxVU0EsMTg5OSwyMjE3ODE5Miw1OTI0NTA1MDAsMTI5MDIxNiwxMDkwNDA2NCwsClVuaXRl
ZCBTdGF0ZXMsVVNBLDE5MDAsMjQ2NTEzOTIsNjI2NTQ0MDAwLDEzOTQ2MjIsMTE1NDE2MDAsLApVbml0ZWQgU3RhdGVzLFVTQSwxOTAxLDI3MTI0NTkyLDY4
MTU0MDcwMCwxNjY2MjIwLDEyODYwNjQwLCwKVW5pdGVkIFN0YXRlcyxVU0EsMTkwMiwzNDg3MDI5MCw3MTU5MzA5NDAsMjE1ODAxMiwxMzY5MjM2OCwsClVu
aXRlZCBTdGF0ZXMsVVNBLDE5MDMsMzk4Mzg2NzAsODQwMDQxNjAwLDI1NDg0NDcsMTQ1MjQwOTYsLApVbml0ZWQgU3RhdGVzLFVTQSwxOTA0LDQ2Nzg5MDg0
LDgxODI3MDQwMCwyNzQ4NjA4LDE1MTYxNTY5LCwxODMzNzUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5MDUsNTM4NTM0NzAsOTEzNDc5MjAwLDM1MTIyMDYsMTcx
NjU4NDAsLDE5ODA0NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkwNiw1MDI3Mzc0NCw5NjQwNDk2NjAsNDQ5NzI5NywxOTAxNjE2MCwsMjEyNzE1MApVbml0ZWQg
U3RhdGVzLFVTQSwxOTA3LDY2Njg4MjY0LDExMTUyNDg1MDAsNDYyOTU1OCwxOTg2NjE0NiwsMjA1MzgwMC4xClVuaXRlZCBTdGF0ZXMsVVNBLDE5MDgsNzE1
NDY5MzAsOTU5NzYyODAwLDQ3MTM0MzEsMTk2NjQ2ODgsLDE4MzM3NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkwOSw3MzIzNjAzMCwxMDY2NjA4ODAwLDU5NTA3
MTMsMjM1MDgyMjQsLDIzNDcyMDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkxMCw4MzkzMTI1MCwxMTYwMzk2MjAwLDY5NTQzNzksMjQ5MDA1NDQsLDIzNDcyMDAK
VW5pdGVkIFN0YXRlcyxVU0EsMTkxMSw4ODE3MDUwMCwxMTQzMTI3NzAwLDcxMTY3MTQsMjUwODc0MDgsLDIyNzM4NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkx
Miw4OTMyMDk5MCwxMjI0Mzk1MzAwLDc0NTk1OTQsMjc0OTQ2NTYsLDIzNDcyMDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkxMywxMDcwNzY3NDAsMTMwNDU3ODMw
MCw4MzIxNzM3LjUsMjg0NTgyODgsLDI0MjA1NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkxNCwxMTQ2MTcyNTAsMTE3MjEzNTcwMCw3OTcyMDkzLDI4OTQ1NjAw
LCwyMjczODUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5MTUsMTIwOTMwMzIwLDEyMTM1OTc0MDAsNzc2MjQxMSwzMDc0MDk2MCwsMjQyMDU1MApVbml0ZWQgU3Rh
dGVzLFVTQSwxOTE2LDEzMzk4MTQ5MCwxMzQ1NDQyNzAwLDgyNjc2MjUsMzY4MzQxOTAsLDI3MTM5NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkxNywxNDc5NDEz
MzAsMTQ4MTY0NDcwMCw4MzY5MDg1LjUsMzg4ODIzNzAsLDI0OTM5MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkxOCwxNTkxODk4MTAsMTU1MTEzNjEwMCw2NDA1
NDU5LDM1MjU4NjcwLCwyMTI3MTUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5MTksMTc5MjE3MjMwLDEyNjEyNTEzMDAsNzI4MTY1MCwzNjQ4MjQ1MCwsMjIwMDUw
MApVbml0ZWQgU3RhdGVzLFVTQSwxOTIwLDIzMDY0NTE0MCwxNDY1NTM0MTAwLDg5Mjk5NzIsMzk3MTA0MzAsLDIzNDcyMDAKVW5pdGVkIFN0YXRlcyxVU0Es
MTkyMSwyNDYzNjAwMzAsMTE0MjY1MTQwMCw4ODE5MTQ3LDMyOTYxMzQ2LCwxNjg3MDUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5MjIsMjgyNDg3ODAwLDExMTI4
OTE2MDAsMTAyNjUwNzEsMzc5NDgxNDgsLDI0MjA1NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkyMywzMzQ0MTMyODAsMTUwOTQ2OTAwMCwxMjMxMDkwNSw1MDEy
NzE4NCwsMjcxMzk1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTI0LDMyNTU1NzQwMCwxMzEzMjA2OTAwLDEzMzc5NjA5LDU2ODI4NjQwLCwyNzEzOTUwClVuaXRl
ZCBTdGF0ZXMsVVNBLDE5MjUsMzQxMjQzNjgwLDEzNDA0ODA5MDAsMTQyMDYzNzEsNTkxNzM3MjQsLDMwODA3MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkyNiwz
NDUyNjYwNTAsMTQ3NzkzMjkwMCwxNDUwNTU0Niw2NTMzNjQ1MCwsMzAwNzM1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTI3LDM5NjIwNjYyMCwxMzc2NjkzMDAw
LDE1MDY2NDMzLDcxOTM4OTgwLCwyOTM0MDAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5MjgsNDA0NDAyMTgwLDEzMzAxNTc2MDAsMTU0MTcxMTgsNzgwNTQwNDAs
LDI5MzQwMDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkyOSw0NDU1OTgyMDAsMTQwNDU4OTcwMCwxNDk5NTY3Miw5NTQ2MjA0MCwsMjg2MDY1MApVbml0ZWQgU3Rh
dGVzLFVTQSwxOTMwLDM5MzQwNjUzMCwxMjM4MTY1NTAwLDE0Mjg1OTc4LDk2Nzg0MzUwLCwyMjczODUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5MzEsMzY2NjIw
NzcwLDEwMjE0ODE3MDAsMTA4MDkzMDksODQyMTM1ODAsLDE4MzM3NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkzMiwzMzc2NTAzMDAsODM0MjMyOTYwLDY2NTU3
MjUsNzc5NTQ4MTAsLDEzMjAzMDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkzMywzNzM3ODY2MjAsODg5ODYxMDAwLDU1ODcwMjEsNzgxMDE4MjAsLDE1NDAzNTAK
VW5pdGVkIFN0YXRlcyxVU0EsMTkzNCwzNzQ5OTMwNjAsOTYzOTIwMjYwLDY4OTk3NDcsODg4MTE5MjAsLDE2MTM3MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTkz
NSw0MDUyMDY0MzAsOTgyMTU1OTcwLDY2ODQzNDEsOTYyOTM4MjAsLDE5ODA0NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTkzNiw0NDgzNTYzNTAsMTE0Mzc5ODMw
MCw5OTQ5NzY3LDEwODgxMzQ3MCwsMjQ5MzkwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTM3LDUxMzYzNTI2MCwxMTQ0OTUxMDAwLDEwMzY4MDkxLDEyMDk0MTU2
MCwsMjcxMzk1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTM4LDQ4MjE5NTkwMCw5MDY3OTM4MDAsOTIzOTE1MiwxMTUzMjA0NTYsLDIyMDA1MDAKVW5pdGVkIFN0
YXRlcyxVU0EsMTkzOSw1MDgxNTc2MzAsMTAyNDk4NzkwMCwxMDc5MjMzNiwxMjQxMjE5NDQsLDI4NjA2NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTk0MCw1NTc2
MjA1MDAsMTE2ODg4MTkwMCwxMTU0ODU2MywxMzM3MDY2OTYsLDMyMjc0MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk0MSw1ODg3MzUxNzAsMTI5NDc2MjQwMCwx
NDUyMDEzOCwxNDE1MzMwMDAsLDQwMzQyNDkuOApVbml0ZWQgU3RhdGVzLFVTQSwxOTQyLDU2NTY5MzI1MCwxNDU4NTk2OTAwLDE2MDg1MzQyLDE1Mzg1NTI4
MCwsNDAzNDI0OS44ClVuaXRlZCBTdGF0ZXMsVVNBLDE5NDMsNjEyOTM1OTQwLDE0NzA4MDI4MDAsMTIwNDEwOTYsMTcxOTUxNTIwLCw0MzI3NjUwClVuaXRl
ZCBTdGF0ZXMsVVNBLDE5NDQsNzAwMTE5NzQwLDE1NDU4NTAyMDAsODAzMTU4NS41LDE4NjU3NDI3MCwsNDMyNzY1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTQ1
LDcyNzM3NzMwMCwxNDIxNDIzMTAwLDkxMTM2NzQsMTk3Njc2NDYwLCwzOTYwOTAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NDYsNzM3MDE4NDMwLDEyOTg3NTgx
MDAsMTQ2NTMwMTUsMjAzMTAzMTgwLCwzOTYwOTAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NDcsNzkxMzQyMjAwLDE0NDkyMzA3MDAsMTY2NDc0OTUsMjI0MDg2
MjQwLCw0NTQ3NzAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NDgsODc0MzQ1MjAwLDE0MzMxOTAzMDAsMTg0MTcwODIsMjUxNzY0NzgwLCw0ODQxMTAwClVuaXRl
ZCBTdGF0ZXMsVVNBLDE5NDksODEzNzIzMTAwLDEwNjMxMDIzMDAsMTg3ODc2NjgsMjY1MDY4NDIwLCw0MTgwOTUwLjIKVW5pdGVkIFN0YXRlcyxVU0EsMTk1
MCw4OTY4MDA2NDAsMTI1NzIzMTkwMCwyMDEyNDU1NCwzMTkyMTg2NjAsNDMxMjE2MTYsNDk4NzgwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTUxLDk2MDY2MTkw
MCwxMjExNTkxMjAwLDIyMDQwNTk0LDM3NjIwOTA2MCw0MjcwMDMxNiw1NTAxMjUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NTIsMTAwMDk2NjcwMCwxMDc0MTY5
NzAwLDIyMjU1NTEyLDQwMjc4NjYwMCw0NTY4MjY4NCw1MzU0NTUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NTMsMTA1MDA3OTAwMCwxMDY1OTU1MTAwLDIzNTIw
ODU0LDQyMzM0MTYwMCw0MzYxOTg1Niw2NDU0ODAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NTQsMTA2MzIxMDU2MCw5MTM0NDg1MDAsMjQxMzA1MzgsNDQzOTk5
MjAwLDM4OTUxOTMwLDU3MjEzMDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk1NSwxMTQ3OTAxODAwLDEwMjYyMTY3NzAsMjY1ODM3NTQsNDc5MTkyNTgwLDQxNjQ1
MDI0LDY5NjgyNTAKVW5pdGVkIFN0YXRlcyxVU0EsMTk1NiwxMjAzNzg1MjAwLDEwNjgzNTI4MDAsMjgzOTAyNzAsNTA1ODg4NDgwLDQ2NTI5MTM2LDcwNDE2
MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk1NywxMTkzNjYzMTAwLDEwMjQwNzg0MDAsMjcwMDU1MTAsNTQwNjQ1OTAwLDQzNTU3NjkwLDY4MjE1NTAKVW5pdGVk
IFN0YXRlcyxVU0EsMTk1OCwxMjIwMDQ2MTAwLDg4ODM0NDEwMCwyNzc1NzkyMiw1NzA3MDEwMDAsMzQwOTcxODQsNjE2MTQwMApVbml0ZWQgU3RhdGVzLFVT
QSwxOTU5LDEyNTg0NTU4MDAsOTEwOTY1NjAwLDMwMjMzOTgyLDU5MzIwODkwMCwzMDc1NTYxNiw4Mjg4NTUwLjUKVW5pdGVkIFN0YXRlcyxVU0EsMTk2MCwx
MjgxNjMwNjAwLDkxNzk1NjYwMCwyODc5ODExMCw2MzAwMjQ3NzAsMzAzMTU5MzYsODU4MTk1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTYxLDEyOTczMTYyMDAs
ODg3NDMxNzQwLDI4Njk4ODEyLDYzNjQxNDgwMCwyODE5ODE0NCw4ODAyMDAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NjIsMTMzNDc0NzUwMCw5MjA4MTQ1MDAs
Mjk4Nzg0OTYsNjc2MzY3MDQwLDIyOTIxOTg0LDkxNjg3NTAKVW5pdGVkIFN0YXRlcyxVU0EsMTk2MywxMzU5MjU2MDAwLDk4NzAwNDYwMCwzMTQzNTE0Miw3
MTg0NTU0MDAsMjA2NTAzMDQsOTY4MjIwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTY0LDEzOTM5MTc0MDAsMTA0OTE3MTgwMCwzMzAyNTY4NCw3NTkwNzA4NTAs
MTg0MTE2MDAsMTA3MDkxMDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk2NSwxNDY0NjkxMzAwLDEwOTA2MTE2MDAsMzMyOTQ3MzgsNzgyNjA0NzQwLDE3MTg3ODI0
LDExMTQ5MjAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NjYsMTUyOTY2MTQwMCwxMTMyNzE0NjAwLDM0NTcyNjE2LDg0MjIwNzA0MCwyMDIzMjYwOCwxMjAyOTQw
MApVbml0ZWQgU3RhdGVzLFVTQSwxOTY3LDE1ODkxOTI0MDAsMTE2MzA1MTQwMCwzMzU5MzkyNCw4ODEyODI3MDAsMjYzODQ0MzgsMTE5NTYwNTAKVW5pdGVk
IFN0YXRlcyxVU0EsMTk2OCwxNjg5NDIyNzAwLDExMzk2MDY1MDAsMzUzMTk3NzYsOTM2MzYwODAwLDI3ODE3MDg4LDEyMzk2MTUwClVuaXRlZCBTdGF0ZXMs
VVNBLDE5NjksMTc4MTYyNzMwMCwxMTU4MDEwOTAwLDM1Njk2OTk2LDEwMTgwODI3MDAsMjgzMTUzOTIsMTM0MjMwNTAKVW5pdGVkIFN0YXRlcyxVU0EsMTk3
MCwyMDM4MDgxNzAwLDExNjgxMDg5MDAsMzQ5MjQxNDgsMTA1OTIxMTEwMCwyNjI0ODg5NiwxMzEyOTY1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTcxLDIwOTcz
MDUwMDAsMTEwNzQyNDgwMCwzNTUwOTc3MiwxMDk2OTI3NDAwLDE1MjYwNTQ3LDEzMDU2MzAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NzIsMjI2OTE3MjIwMCwx
MTI0ODYxNzAwLDM2NTIzNDkyLDExMTU2NzI0MDAsMTMzMDc2MzcsMTM0OTY0MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk3MywyMzIxNDg0MzAwLDEyMTg3MjA4
MDAsMzY5MTcwNTYsMTE3MzMzMzAwMCwxMzE4Njc0NiwxNDA4MzIwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTc0LDIyMjEwNjMyMDAsMTE5NTk0NTMwMCwzNjgw
NjMxNiwxMTMxMzcxMzAwLDg4OTk4NTYsMTQzNzY2MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk3NSwyMTU4NTYxMzAwLDExNzc4MDA0MDAsMzA0NjMyMjQsMTAy
Nzc1NTUwMCw3MTExODI0LDEyNzYyOTAwClVuaXRlZCBTdGF0ZXMsVVNBLDE5NzYsMjI4NzEyNzgwMCwxMjQ2NzA3MjAwLDMyMzcwMDIyLDEwMzQ0OTg3NTAs
NzMzODk5MiwxMzQ5NjQwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTc3LDI0MDYyNDQwMDAsMTI3NzU4MjYwMCwzMzk3ODkwNCwxMDExMTU2ODYwLDcyNjIwNDgs
MTMyNzYzNTAKVW5pdGVkIFN0YXRlcyxVU0EsMTk3OCwyNTAxNjI0MDAwLDEzMDIzMjczMDAsMzU2MTU4MjAsMTAzNjA3OTEwMCw4MTIzMDg4LDEzNTY5NzUw
ClVuaXRlZCBTdGF0ZXMsVVNBLDE5NzksMjM5MjM5NDgwMCwxMzk4NjU3MjAwLDM1OTQwNDAwLDEwNTg5NzAyNDAsODg2Njg3MywxMzkzNjUwMApVbml0ZWQg
U3RhdGVzLFVTQSwxOTgwLDIyMDcyMzYwMDAsMTQyOTg0ODQwMCwzMjkyMTY5MiwxMDQwOTA1NTQwLDY2OTA0NjQsMTI2MTYyMDAKVW5pdGVkIFN0YXRlcyxV
U0EsMTk4MSwyMDM2Njc2MTAwLDE0NTY2NDk1MDAsMzE5NjIxNTYsMTAwMDU3MDQzMCw1MjMyMTkyLDEyNTQyODUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5ODIs
MTkzMjkzMTcwMCwxNDA0NTUyODAwLDI4NDQwNzcwLDkzMTYxNDAwMCw0OTkwMzY4LDkzODg4MDAKVW5pdGVkIFN0YXRlcyxVU0EsMTk4MywxOTQwMzgwNDAw
LDE0NzYzMDY4MDAsMzA0Mzk1MDQsODg0MjYyNjAwLDUwODkzMDAsOTkwMjI1MApVbml0ZWQgU3RhdGVzLFVTQSwxOTg0LDE5NTM2NzI3MDAsMTU1MjA3Mjgw
MCwzMjkxNDk3NCw5MjQzNjc1NTAsNTc4OTExNSwxMDU2MjQwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTg1LDE5NTQ1MDY0MDAsMTU5ODUyODQwMCwzMTcwODAw
Niw4OTUxMTk5MDAsNTEwMDI4OCwxMDQxNTcwMApVbml0ZWQgU3RhdGVzLFVTQSwxOTg2LDIwMzYyMzMxMDAsMTU3NzE4MDgwMCwzMjg1MzYxNCw4MzcwMTYy
NjAsNTI0MzE4MCw5NjA4ODUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5ODcsMjA4Mjg1NDgwMCwxNjYxNjE4NDAwLDMyOTc1NzYyLDg5NzM2MDMwMCw2NjMxODQw
LDEwNDg5MDUwClVuaXRlZCBTdGF0ZXMsVVNBLDE5ODgsMjE3MjQwNzYwMCwxNzM3OTU2NzAwLDMzMjQ4MDI4LDkzNTE1MTMwMCw3NjQzMDk4LDExMzY5MjUw
ClVuaXRlZCBTdGF0ZXMsVVNBLDE5ODksMjE2MzY5NzcwMCwxNzUxNzgxMjAwLDMzMjY0MjA2LDk5MjQ1NjMwMCw3NTk1NDcyLDExNDQyNjAwClVuaXRlZCBT
dGF0ZXMsVVNBLDE5OTAsMjEzOTM0NTcwMCwxNzgxODI0MTAwLDMzNDg0MTQyLDEwMDQ5NzQ4NTAsNDI2Njc1ODAsMTI5NDY0NjY0ClVuaXRlZCBTdGF0ZXMs
VVNBLDE5OTEsMjA4MjA5OTEwMCwxNzY0MjIwNzAwLDMyNzM2NDY2LDEwMjMzMzE2MDAsNDE4NDc4NjQsMTMyMTc2OTg0ClVuaXRlZCBTdGF0ZXMsVVNBLDE5
OTIsMjEzMDQ4ODgwMCwxNzgwNTk4OTAwLDMyOTkyOTc4LDEwNTg1OTI1MDAsNDE0NTU1NjQsMTM0NjQwODYwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTMsMjE0
NDE1NjUwMCwxODQ3MjkxOTAwLDM0ODM3OTc2LDEwOTAxNjU4MDAsNDEwOTkwMjAsMTIxOTY5MjkwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTQsMjE4MTg0Mzcw
MCwxODU1OTM3ODAwLDM2MzEwNDMwLDExMTQzMzM3MDAsNDEwNTAxMDAsMTMyMzg3MzkwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTUsMjE3NDMwMzUwMCwxODc1
Nzg2OTAwLDM3MDc1MjgwLDExNjMxODYyMDAsMzkyNzYyNzYsMTM2MjA5MjgwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTYsMjI1NzAyMjcwMCwxOTUxODE1MDAw
LDM3MzA4ODk2LDExODEzMTg1MDAsMzc4ODU2MjgsMTM1NzQzMTgwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTcsMjI3MzgxNTYwMCwxOTkyNzIzMzAwLDM4NTYw
NzQ0LDExODY3MTY3MDAsMzc5MTA4MjAsMTQzNDE2ODYwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTgsMjMyMDA4MDEwMCwyMDE4NjA5MjAwLDM5NDYwODcwLDEx
NjU1NTg5MDAsMzU1ODAyNjAsMTU3MDcwOTYwClVuaXRlZCBTdGF0ZXMsVVNBLDE5OTksMjM3NDIwMTMwMCwyMDEzMTIwMDAwLDQwMjM4NzI0LDExNjg3NTkw
MDAsMzU0OTQyOTIsMTY4NDY3ODkwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMDAsMjQ0MDQwNDcwMCwyMTIyMzM5ODAwLDQxNDQ1MzA4LDEyMjc1Mjg3MDAsMzU5
ODM3ODQsMTU1NDU1MTAwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMDEsMjQ0ODYyMjAwMCwyMDU0NzI4NjAwLDQxNjEzMzY0LDExNzMxNjIyMDAsMzU3MDcxMzIs
MTQ5NzI4NzgwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMDIsMjQ0MzgyMzAwMCwyMDYwMDcyMTAwLDQzMTYzODcwLDEyMTIwODY3MDAsMzYwOTA4NjAsMTUyNTgy
NzgwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMDMsMjUwNTEzNDMwMCwyMTAwMzIzMjAwLDQzMzQ5MTA0LDExNzY1NTEyMDAsMzYxMzc2NDAsMTQ0OTQ0MjYwClVu
aXRlZCBTdGF0ZXMsVVNBLDIwMDQsMjU3MTEwNzMwMCwyMTE1MjkyNzAwLDQ1ODg1NTgwLDExNzc1Mjc3MDAsMzY4MjM4NDAsMTY0OTA3NjMwClVuaXRlZCBT
dGF0ZXMsVVNBLDIwMDUsMjU4NDEyOTgwMCwyMTM5OTU0MzAwLDQ2MTk0MTI0LDExNjA2NjA1MDAsMzcwMTU0NzYsMTU4OTQ4ODIwClVuaXRlZCBTdGF0ZXMs
VVNBLDIwMDYsMjU1MDU0MDAwMCwyMTAzODcxMTAwLDQ2ODUwNzQ0LDExNDY3MTQxMDAsMzgwMDMyMzYsMTU5MzQ2OTgwClVuaXRlZCBTdGF0ZXMsVVNBLDIw
MDcsMjUzODk2ODAwMCwyMTMwMTY5MjAwLDQ1NTA4ODgwLDEyMjE1OTMwMDAsMzg0ODA4MzYsMTQ2ODIyNTYwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMDgsMjM3
MDYyMjcwMCwyMDk2ODg3OTAwLDQxNDE1NjUyLDEyMjk3MzMyMDAsNDAwNTE1NTAsMTQwNTA5NTcwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMDksMjI0Mzc4OTgw
MCwxODQyMzkxNzAwLDI5NjE0NjQ0LDEyMTE5NjU2MDAsMzg2MDIwMTAsMTE5NzQwNDkwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTAsMjI1ODI5MjcwMCwxOTQ2
NTEwMjAwLDMxNDQ5MjM2LDEyNjY3NzgxMDAsNDA4OTU1OTYsMTI1MzIzOTgwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTEsMjIxMjY4NTgwMCwxODQwMjQzMDAw
LDMyMjA4MzU4LDEyODc1ODg1MDAsNDM3NjQzNDQsMTIyNDgzMjEwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTIsMjE1MjI0NDIwMCwxNjI1NDI3MjAwLDM1Mjcw
MzQ0LDEzNDQ3MTg1MDAsNDc0OTIyNzAsMTI2MzE0Nzc2ClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTMsMjE3ODQxNzIwMCwxNjg0NjcxNzAwLDM2MzY5MjI4LDEz
ODExODI4MDAsNTMwMjI1MDQsMTM5NzcyNjIwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTQsMjIxMDU2OTIwMCwxNjgyMTYyMjAwLDM5NDM5MDIwLDE0MTEzNTQ5
MDAsNTg3NDgyMjgsMTI5MTExNzQwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTUsMjIzNTk4ODAwMCwxNDQ2OTkwMjAwLDM5OTA3MjkyLDE0NDQwNDcyMDAsNjA1
NzQyMDAsMTQwOTg5NzAwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTYsMjI1MjI3MDMwMCwxMzE5OTkzMTAwLDM5NDM5MDIwLDE0NTExNDU5MDAsNTE2NTQ4MDQs
MTMwODU4NzkwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTcsMjI1OTE1NjcwMCwxMjc3MDQ5MDAwLDQwMzIzNTM2LDE0MjQ5OTY0MDAsNTUyNzE3NzIsMTM4NjIw
MzcwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTgsMjMwNTQ0MTMwMCwxMjI0Mzc5OTAwLDM4OTcwNzQ0LDE1Nzc3MTYyMDAsNjc5MjE3MDAsMTQ2ODA2NDUwClVu
aXRlZCBTdGF0ZXMsVVNBLDIwMTksMjI5OTgyMDAwMCwxMDQzODk2MjYwLDQwODk1ODcwLDE2MzIwNTU3MDAsODQ1MTk1NDAsMTM0NzI0ODAwClVuaXRlZCBT
dGF0ZXMsVVNBLDIwMjAsMTk5MTU3ODgwMCw4NTM5MjQ3NDAsNDA2ODc3NDgsMTYxMjUwNDAwMCw2NjA4NDk4NCwxMjUxNzMzMjAKVW5pdGVkIFN0YXRlcyxV
U0EsMjAyMSwyMTgyNjA1NjAwLDk3OTUwMzU1MCw0MTMxMjExMCwxNjE3NDIzOTAwLDYwMjIwNTg4LDEzOTA0NTUyMApVbml0ZWQgU3RhdGVzLFVTQSwyMDIy
LDIxOTg5MzcwMDAsOTE2MzQzMDQwLDQxODg0NDQ0LDE3MDczOTE0MDAsNTg4ODIwMjgsMTMxOTY1MDIwClVuaXRlZCBTdGF0ZXMsVVNBLDIwMjMsMjE5OTEx
MTAwMCw3NTg1NzIwMDAsNDA2MzU3MTYsMTcyMzQyNDEwMCw2MTMyODE2MCwxMzUzMzU4OTAKVW5pdGVkIFN0YXRlcyxVU0EsMjAyNCwyMTg5MTYxMDAwLDcz
Mjg4NjMwMCwzNzI3MDY1NiwxNzQ4MTM3NzAwLDYxMzI4MTYwLDEzNTMzNTg5MA=="""
)
_DATA["us_percapita_ghg"] = (
"""RW50aXR5LENvZGUsWWVhcixQZXIgY2FwaXRhIGdyZWVuaG91c2UgZ2FzIGVtaXNzaW9ucyBpbmNsdWRpbmcgbGFuZCB1c2UKVW5pdGVkIFN0YXRlcyxVU0Es
MTg1MCwzMi43OTg5MTYKVW5pdGVkIFN0YXRlcyxVU0EsMTg1MSwzNC4yNDExNDYKVW5pdGVkIFN0YXRlcyxVU0EsMTg1MiwzNC4yNjc3MgpVbml0ZWQgU3Rh
dGVzLFVTQSwxODUzLDM0Ljk0NzU3NQpVbml0ZWQgU3RhdGVzLFVTQSwxODU0LDM0Ljk3MDMKVW5pdGVkIFN0YXRlcyxVU0EsMTg1NSwzNC42NzY0OTUKVW5p
dGVkIFN0YXRlcyxVU0EsMTg1NiwzNC4zMDc4NzcKVW5pdGVkIFN0YXRlcyxVU0EsMTg1NywzNC4xMTk2NwpVbml0ZWQgU3RhdGVzLFVTQSwxODU4LDM0LjA4
NzYKVW5pdGVkIFN0YXRlcyxVU0EsMTg1OSwzNC4zMjk3MwpVbml0ZWQgU3RhdGVzLFVTQSwxODYwLDMxLjIxMzU0OQpVbml0ZWQgU3RhdGVzLFVTQSwxODYx
LDI4LjI2NDgwMQpVbml0ZWQgU3RhdGVzLFVTQSwxODYyLDI3LjUzNTQ3NQpVbml0ZWQgU3RhdGVzLFVTQSwxODYzLDI3LjIwMTQyMgpVbml0ZWQgU3RhdGVz
LFVTQSwxODY0LDI2LjA5NzkyOQpVbml0ZWQgU3RhdGVzLFVTQSwxODY1LDI0LjgyMzU5OQpVbml0ZWQgU3RhdGVzLFVTQSwxODY2LDIzLjk2NDQ0MQpVbml0
ZWQgU3RhdGVzLFVTQSwxODY3LDIzLjk3MTIwNwpVbml0ZWQgU3RhdGVzLFVTQSwxODY4LDIzLjM4NjIKVW5pdGVkIFN0YXRlcyxVU0EsMTg2OSwyMi42NTky
NApVbml0ZWQgU3RhdGVzLFVTQSwxODcwLDI4LjI5MjA2OApVbml0ZWQgU3RhdGVzLFVTQSwxODcxLDMwLjQ0OTk4ClVuaXRlZCBTdGF0ZXMsVVNBLDE4NzIs
MzEuODk1MjgKVW5pdGVkIFN0YXRlcyxVU0EsMTg3MywzMy4xODUzNwpVbml0ZWQgU3RhdGVzLFVTQSwxODc0LDMzLjYzMTk0NwpVbml0ZWQgU3RhdGVzLFVT
QSwxODc1LDMzLjgzNzgyMgpVbml0ZWQgU3RhdGVzLFVTQSwxODc2LDMzLjkxNDg3NQpVbml0ZWQgU3RhdGVzLFVTQSwxODc3LDM0LjU4Mjg5ClVuaXRlZCBT
dGF0ZXMsVVNBLDE4NzgsMzQuNzg2MDg3ClVuaXRlZCBTdGF0ZXMsVVNBLDE4NzksMzUuNzcxNjIKVW5pdGVkIFN0YXRlcyxVU0EsMTg4MCwzMy41MDAyNwpV
bml0ZWQgU3RhdGVzLFVTQSwxODgxLDMxLjYwNTg1ClVuaXRlZCBTdGF0ZXMsVVNBLDE4ODIsMzEuNjgxMjc4ClVuaXRlZCBTdGF0ZXMsVVNBLDE4ODMsMzEu
NzkxMTYyClVuaXRlZCBTdGF0ZXMsVVNBLDE4ODQsMzEuMTc4MgpVbml0ZWQgU3RhdGVzLFVTQSwxODg1LDMwLjI4Njc4MQpVbml0ZWQgU3RhdGVzLFVTQSwx
ODg2LDI5Ljk0NjYzNApVbml0ZWQgU3RhdGVzLFVTQSwxODg3LDMwLjEwMDAwNApVbml0ZWQgU3RhdGVzLFVTQSwxODg4LDMwLjQ4NTExClVuaXRlZCBTdGF0
ZXMsVVNBLDE4ODksMjguNzk2ODMxClVuaXRlZCBTdGF0ZXMsVVNBLDE4OTAsMzAuNjczNTg0ClVuaXRlZCBTdGF0ZXMsVVNBLDE4OTEsMzAuOTY1MTk3ClVu
aXRlZCBTdGF0ZXMsVVNBLDE4OTIsMzAuNzAzOTcKVW5pdGVkIFN0YXRlcyxVU0EsMTg5MywzMC43MDMwMjQKVW5pdGVkIFN0YXRlcyxVU0EsMTg5NCwyOS43
MjA5MDEKVW5pdGVkIFN0YXRlcyxVU0EsMTg5NSwzMC4wNTgwMjMKVW5pdGVkIFN0YXRlcyxVU0EsMTg5NiwyOS43MDA4NDIKVW5pdGVkIFN0YXRlcyxVU0Es
MTg5NywyOS42ODAzMzgKVW5pdGVkIFN0YXRlcyxVU0EsMTg5OCwzMC4xMzI0OQpVbml0ZWQgU3RhdGVzLFVTQSwxODk5LDMxLjA3ODEwMgpVbml0ZWQgU3Rh
dGVzLFVTQSwxOTAwLDMwLjMxNDUwOApVbml0ZWQgU3RhdGVzLFVTQSwxOTAxLDMwLjEwNjczNwpVbml0ZWQgU3RhdGVzLFVTQSwxOTAyLDMwLjE2Njc0ClVu
aXRlZCBTdGF0ZXMsVVNBLDE5MDMsMzEuMTUyNDAzClVuaXRlZCBTdGF0ZXMsVVNBLDE5MDQsMzAuNjUxNzEKVW5pdGVkIFN0YXRlcyxVU0EsMTkwNSwzMS4z
NzQxOQpVbml0ZWQgU3RhdGVzLFVTQSwxOTA2LDMxLjMxODEwMgpVbml0ZWQgU3RhdGVzLFVTQSwxOTA3LDMyLjk2NzAyMgpVbml0ZWQgU3RhdGVzLFVTQSwx
OTA4LDMwLjQ3NDYzClVuaXRlZCBTdGF0ZXMsVVNBLDE5MDksMzEuNDI4NTIKVW5pdGVkIFN0YXRlcyxVU0EsMTkxMCwzMS43MjE4NzgKVW5pdGVkIFN0YXRl
cyxVU0EsMTkxMSwzMC45MDI4OTcKVW5pdGVkIFN0YXRlcyxVU0EsMTkxMiwzMC43ODQ4NzIKVW5pdGVkIFN0YXRlcyxVU0EsMTkxMywzMS41MTQ2NDgKVW5p
dGVkIFN0YXRlcyxVU0EsMTkxNCwyOS42NzE5MDcKVW5pdGVkIFN0YXRlcyxVU0EsMTkxNSwyOS41MjQwNTcKVW5pdGVkIFN0YXRlcyxVU0EsMTkxNiwzMC41
MzgxMzYKVW5pdGVkIFN0YXRlcyxVU0EsMTkxNywzMS43MzMxMDcKVW5pdGVkIFN0YXRlcyxVU0EsMTkxOCwzMi4yMTY5MTUKVW5pdGVkIFN0YXRlcyxVU0Es
MTkxOSwyOC41ODI2NDcKVW5pdGVkIFN0YXRlcyxVU0EsMTkyMCwzMS4wMTkzMjEKVW5pdGVkIFN0YXRlcyxVU0EsMTkyMSwyNy40ODY5OTYKVW5pdGVkIFN0
YXRlcyxVU0EsMTkyMiwyNy4wNzkxMTkKVW5pdGVkIFN0YXRlcyxVU0EsMTkyMywzMS4yMTQ4NzYKVW5pdGVkIFN0YXRlcyxVU0EsMTkyNCwyOC42MDU4NDYK
VW5pdGVkIFN0YXRlcyxVU0EsMTkyNSwyOC43NzgxMTQKVW5pdGVkIFN0YXRlcyxVU0EsMTkyNiwyOS41ODMwMjkKVW5pdGVkIFN0YXRlcyxVU0EsMTkyNywy
OC40OTEwMjIKVW5pdGVkIFN0YXRlcyxVU0EsMTkyOCwyNy42NTgzOTQKVW5pdGVkIFN0YXRlcyxVU0EsMTkyOSwyOC4zNTkxOQpVbml0ZWQgU3RhdGVzLFVT
QSwxOTMwLDI1LjQwMDY3NQpVbml0ZWQgU3RhdGVzLFVTQSwxOTMxLDIyLjU2MTgzClVuaXRlZCBTdGF0ZXMsVVNBLDE5MzIsMTkuODM4MzIyClVuaXRlZCBT
dGF0ZXMsVVNBLDE5MzMsMjAuMjU2NjEKVW5pdGVkIFN0YXRlcyxVU0EsMTkzNCwyMC44MjA1NjQKVW5pdGVkIFN0YXRlcyxVU0EsMTkzNSwyMC44MjQzNApV
bml0ZWQgU3RhdGVzLFVTQSwxOTM2LDIyLjUyNzIyMgpVbml0ZWQgU3RhdGVzLFVTQSwxOTM3LDIyLjc5MTU1NQpVbml0ZWQgU3RhdGVzLFVTQSwxOTM4LDIw
LjA3NTM2ClVuaXRlZCBTdGF0ZXMsVVNBLDE5MzksMjEuMDY1OTgKVW5pdGVkIFN0YXRlcyxVU0EsMTk0MCwyMi45NjQ0MjQKVW5pdGVkIFN0YXRlcyxVU0Es
MTk0MSwyNC4zMzAyNDgKVW5pdGVkIFN0YXRlcyxVU0EsMTk0MiwyNS40MzMyNjIKVW5pdGVkIFN0YXRlcyxVU0EsMTk0MywyNS43ODgzMzIKVW5pdGVkIFN0
YXRlcyxVU0EsMTk0NCwyNi45NDI1MDEKVW5pdGVkIFN0YXRlcyxVU0EsMTk0NSwyNi4wMDEwODEKVW5pdGVkIFN0YXRlcyxVU0EsMTk0NiwyNC45NTg0OTgK
VW5pdGVkIFN0YXRlcyxVU0EsMTk0NywyNi41MDI3OTgKVW5pdGVkIFN0YXRlcyxVU0EsMTk0OCwyNi45Njk4ODEKVW5pdGVkIFN0YXRlcyxVU0EsMTk0OSwy
My4zNDE2NzUKVW5pdGVkIFN0YXRlcyxVU0EsMTk1MCwyNC4yOTUyODgKVW5pdGVkIFN0YXRlcyxVU0EsMTk1MSwyNC4yMTcyNDkKVW5pdGVkIFN0YXRlcyxV
U0EsMTk1MiwyMy4yMTA0MDMKVW5pdGVkIFN0YXRlcyxVU0EsMTk1MywyMy4yMDc4MzIKVW5pdGVkIFN0YXRlcyxVU0EsMTk1NCwyMi4wNDI4MQpVbml0ZWQg
U3RhdGVzLFVTQSwxOTU1LDIzLjIyOTM2NgpVbml0ZWQgU3RhdGVzLFVTQSwxOTU2LDIzLjczNTYwMQpVbml0ZWQgU3RhdGVzLFVTQSwxOTU3LDIzLjEzNzgz
OApVbml0ZWQgU3RhdGVzLFVTQSwxOTU4LDIyLjE4MTc1OQpVbml0ZWQgU3RhdGVzLFVTQSwxOTU5LDIyLjEwNzI4MwpVbml0ZWQgU3RhdGVzLFVTQSwxOTYw
LDIyLjE4MzY3ClVuaXRlZCBTdGF0ZXMsVVNBLDE5NjEsMjEuMzY4NjgzClVuaXRlZCBTdGF0ZXMsVVNBLDE5NjIsMjIuMzM1NTMKVW5pdGVkIFN0YXRlcyxV
U0EsMTk2MywyMi40MDczODUKVW5pdGVkIFN0YXRlcyxVU0EsMTk2NCwyMi44NDQ0NzEKVW5pdGVkIFN0YXRlcyxVU0EsMTk2NSwyMy4yMDk4MTYKVW5pdGVk
IFN0YXRlcyxVU0EsMTk2NiwyMy44NjA4MTMKVW5pdGVkIFN0YXRlcyxVU0EsMTk2NywyNi4xNTE1NDYKVW5pdGVkIFN0YXRlcyxVU0EsMTk2OCwyNy45ODg2
NjcKVW5pdGVkIFN0YXRlcyxVU0EsMTk2OSwyNy40MzQ1MTUKVW5pdGVkIFN0YXRlcyxVU0EsMTk3MCwyOC4wMTU4MjEKVW5pdGVkIFN0YXRlcyxVU0EsMTk3
MSwyNy42MDkxNTYKVW5pdGVkIFN0YXRlcyxVU0EsMTk3MiwyOC4yMjk5NDgKVW5pdGVkIFN0YXRlcyxVU0EsMTk3MywyOC44MTMzODUKVW5pdGVkIFN0YXRl
cyxVU0EsMTk3NCwyNy43MDc4MzIKVW5pdGVkIFN0YXRlcyxVU0EsMTk3NSwyNi40OTU2ODYKVW5pdGVkIFN0YXRlcyxVU0EsMTk3NiwyNy4yNDM3NgpVbml0
ZWQgU3RhdGVzLFVTQSwxOTc3LDI3LjkxODcyNApVbml0ZWQgU3RhdGVzLFVTQSwxOTc4LDI4LjAwMjY0NApVbml0ZWQgU3RhdGVzLFVTQSwxOTc5LDI3Ljgz
MDgxClVuaXRlZCBTdGF0ZXMsVVNBLDE5ODAsMjYuODQzOTI1ClVuaXRlZCBTdGF0ZXMsVVNBLDE5ODEsMjUuNzA4MzQ1ClVuaXRlZCBTdGF0ZXMsVVNBLDE5
ODIsMjQuMzgyNjgKVW5pdGVkIFN0YXRlcyxVU0EsMTk4MywyNC4xNTkwODQKVW5pdGVkIFN0YXRlcyxVU0EsMTk4NCwyNC41ODk5MDkKVW5pdGVkIFN0YXRl
cyxVU0EsMTk4NSwyNC4zOTIyMTgKVW5pdGVkIFN0YXRlcyxVU0EsMTk4NiwyMy45MjY1NTIKVW5pdGVkIFN0YXRlcyxVU0EsMTk4NywyNC41NjMxMzEKVW5p
dGVkIFN0YXRlcyxVU0EsMTk4OCwyNS4yMzA0OTIKVW5pdGVkIFN0YXRlcyxVU0EsMTk4OSwyNS4yNjEyMjUKVW5pdGVkIFN0YXRlcyxVU0EsMTk5MCwyNS42
MTE1NjMKVW5pdGVkIFN0YXRlcyxVU0EsMTk5MSwyNC45Mjg1NjQKVW5pdGVkIFN0YXRlcyxVU0EsMTk5MiwyNC44OTkzNTEKVW5pdGVkIFN0YXRlcyxVU0Es
MTk5MywyNC45NTAzODYKVW5pdGVkIFN0YXRlcyxVU0EsMTk5NCwyNS4xODYzMTQKVW5pdGVkIFN0YXRlcyxVU0EsMTk5NSwyNC43NjUwOTMKVW5pdGVkIFN0
YXRlcyxVU0EsMTk5NiwyNS4xMjk4MDUKVW5pdGVkIFN0YXRlcyxVU0EsMTk5NywyNS4xNzgxMTQKVW5pdGVkIFN0YXRlcyxVU0EsMTk5OCwyNS4wODYxOTEK
VW5pdGVkIFN0YXRlcyxVU0EsMTk5OSwyNS4xNjIzNDYKVW5pdGVkIFN0YXRlcyxVU0EsMjAwMCwyNS42Mzk5NTQKVW5pdGVkIFN0YXRlcyxVU0EsMjAwMSwy
NC41NDA0MjYKVW5pdGVkIFN0YXRlcyxVU0EsMjAwMiwyNC41ODg5ODIKVW5pdGVkIFN0YXRlcyxVU0EsMjAwMywyNC45OTIyMjYKVW5pdGVkIFN0YXRlcyxV
U0EsMjAwNCwyNC45ODI5NgpVbml0ZWQgU3RhdGVzLFVTQSwyMDA1LDI1LjEyMTM0NgpVbml0ZWQgU3RhdGVzLFVTQSwyMDA2LDIzLjY0OTY5MwpVbml0ZWQg
U3RhdGVzLFVTQSwyMDA3LDIzLjc5MjEyNgpVbml0ZWQgU3RhdGVzLFVTQSwyMDA4LDIzLjE5Njc5NQpVbml0ZWQgU3RhdGVzLFVTQSwyMDA5LDIxLjM1NTI1
ClVuaXRlZCBTdGF0ZXMsVVNBLDIwMTAsMjEuOTM2MTYKVW5pdGVkIFN0YXRlcyxVU0EsMjAxMSwyMC43Njg5NzQKVW5pdGVkIFN0YXRlcyxVU0EsMjAxMiwy
MC4xNDYzMgpVbml0ZWQgU3RhdGVzLFVTQSwyMDEzLDIwLjI3MDA2MQpVbml0ZWQgU3RhdGVzLFVTQSwyMDE0LDIwLjQ1OTQ2MwpVbml0ZWQgU3RhdGVzLFVT
QSwyMDE1LDE5LjY0OTY5NApVbml0ZWQgU3RhdGVzLFVTQSwyMDE2LDE5LjE2NDY3NQpVbml0ZWQgU3RhdGVzLFVTQSwyMDE3LDE5LjAyMjA4MwpVbml0ZWQg
U3RhdGVzLFVTQSwyMDE4LDE5LjMwMDQzOApVbml0ZWQgU3RhdGVzLFVTQSwyMDE5LDE4LjgzMDU5ClVuaXRlZCBTdGF0ZXMsVVNBLDIwMjAsMTYuOTY0NzY3
ClVuaXRlZCBTdGF0ZXMsVVNBLDIwMjEsMTguMDI0MTc0ClVuaXRlZCBTdGF0ZXMsVVNBLDIwMjIsMTguMTU3NzkxClVuaXRlZCBTdGF0ZXMsVVNBLDIwMjMs
MTcuNjk2ODE1ClVuaXRlZCBTdGF0ZXMsVVNBLDIwMjQsMTcuNTI3MDQ4"""
)
_DATA["share_cumulative_oil"] = (
"""..."""
)
_DATA["share_cumulative_coal"] = (
"""..."""
)
_DATA["share_cumulative_cement"] = (
"""...
OTkuOTk5OTk="""
)

import base64, io
def load_dataset(name):
    """Load one of the embedded datasets by name into a DataFrame."""
    csv = base64.b64decode(_DATA[name]).decode("utf-8")
    return pd.read_csv(io.StringIO(csv))

print("datasets available:", sorted(_DATA))

## 1 · Global context — who emits the most CO₂ per person?
Two peer groups on the **same color scale**; ▲/▼ show the change since 2014.

In [ ]:
df = load_dataset('co2_per_capita')
df = df.pivot_table(index='Entity', columns='Year', values='CO₂ emissions per capita')
df.columns = [f'y{c}' for c in df.columns]
df = df.reset_index()

OIL  = ['Qatar','Kuwait','Brunei','Bahrain','Trinidad and Tobago',
        'Saudi Arabia','United Arab Emirates','Oman']
ECON = ['United States','Russia','North America','China',
        'European Union (27)','World','United Kingdom','India']
vmax  = df[df.Entity.isin(OIL + ECON)].y2024.max()
world = df.loc[df.Entity == 'World', 'y2024'].iloc[0]
pick  = lambda names: df[df.Entity.isin(names)]

ranked_bar(pick(OIL), category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t', reference=world, reference_label='World average',
           title='A few small, oil-rich nations emit the most CO₂ per person',
           subtitle='Tonnes of CO₂ per person, 2024')
plt.show()

In [ ]:
ranked_bar(pick(ECON), category='Entity', value='y2024', compare='y2014',
           vmax=vmax, unit='t',
           title='Among big economies, the US still emits the most per person',
           subtitle='Tonnes of CO₂ per person, 2024 — same scale as the oil producers')
plt.show()

## 2 · What fuels drive it? — per-capita CO₂ by source
The same totals, split into **coal / oil / gas / flaring / cement / other** — a replica of the Our World in Data chart.

In [ ]:
d = load_dataset('percapita_co2_by_source')
SEG = ['Coal','Oil','Gas','Flaring','Cement','Other industry']
OWID = {'Coal':'#6d6e70','Oil':'#c14b62','Gas':'#8c6bb1',
        'Flaring':'#c8a45c','Cement':'#2f8e7f','Other industry':'#6d8fc5'}
tonnes = lambda v: f'{v:.0f} t' if v >= 10 else f'{v:.1f} t'   # 6.4 t but 34 t

stacked_bar(d, category='Entity', segments=SEG, colors=OWID,
            value_fmt=tonnes, seg_label_min=0.05,
            title='Per capita CO₂ emissions by source, 2024',
            figsize=(11, 8))
plt.show()

## 3 · The US over time — CO₂ by fuel (the hero)
A century-long stacked area: **coal → oil → gas**, annotated with the events that shaped it.

In [ ]:
h = load_dataset('us_co2_by_fuel')
FUELS = ['Coal','Oil','Gas','Cement','Flaring','Other industry']
for f in FUELS:
    h[f] = pd.to_numeric(h[f], errors='coerce') / 1e9   # tonnes -> billion tonnes

EVENTS = [{'year':1932,'label':'1932\nGreat Depression','y':0.42},
          {'year':1945,'label':'1945\nWWII','y':0.72},
          {'year':1973,'label':'1973\nOil shock','y':0.9},
          {'year':2007,'label':'2007\nemissions peak','y':0.98},
          {'year':2020,'label':'2020\nCOVID','y':0.62}]

stacked_area(h, x='Year', series=FUELS, y_label='Billion tonnes CO₂ / year',
             title='Coal gave way to oil and gas',
             subtitle='US CO₂ emissions by fuel or industry, 1800–2024',
             events=EVENTS)
plt.show()

## 4 · Beyond CO₂ — per-capita greenhouse gases
Total greenhouse gases (CO₂ **+ methane + nitrous oxide**), in CO₂-equivalents. The US has roughly halved its per-person footprint — but it is still ~18 t/person.

In [ ]:
g = load_dataset("us_percapita_ghg")
col = [c for c in g.columns if "greenhouse" in c.lower()][0]
x, y = g.Year.to_numpy(), g[col].to_numpy()

fig, ax = plt.subplots(figsize=(9, 4.5))
c = series_color(0)
ax.plot(x, y, color=c, linewidth=2.2, solid_capstyle="round")
ax.fill_between(x, y, color=c, alpha=0.08)
ax.set_ylim(0, y.max()*1.08); ax.set_xlim(x.min(), x.max())
ax.annotate(f"{y[-1]:.0f} t", xy=(x[-1], y[-1]), xytext=(6, 0),
            textcoords="offset points", va="center", fontweight="bold", color=c)
for s in ("top","right"): ax.spines[s].set_visible(False)
ax.tick_params(length=0)
ax.set_title("Even as CO\u2082 fell, the US still emits ~18 t per person",
             loc="left", fontsize=14, fontweight="bold")
plt.show()

## 5 · The historical bill — US share of cumulative CO₂
Of **all** the CO₂ the world has ever emitted from each source, how much came from the United States?

In [ ]:
rows = []
for fuel, name in [('Oil','share_cumulative_oil'),
                   ('Coal','share_cumulative_coal'),
                   ('Cement','share_cumulative_cement')]:
    dd = load_dataset(name)
    col = [c for c in dd.columns if 'Share' in c][0]
    rows.append({'Fuel': fuel,
                 'share': dd[(dd.Entity=='United States') & (dd.Year==2024)][col].iloc[0]})

ranked_bar(pd.DataFrame(rows), category='Fuel', value='share', value_fmt='{:.0f}%',
           title='A quarter of all oil CO₂ ever, from one country',
           subtitle="US share of the world's cumulative CO₂, by source")
plt.show()

## 6 · The full poster
Everything above, composed onto one canvas.

In [ ]:
# The full poster — hero + two supporting panels on one canvas
import matplotlib.pyplot as plt

FUELS = ["Coal","Oil","Gas","Cement","Flaring","Other industry"]
EVENTS = [{"year":1932,"label":"1932\nGreat Depression","y":0.42},
          {"year":1945,"label":"1945\nWWII","y":0.72},
          {"year":1973,"label":"1973\nOil shock","y":0.9},
          {"year":2007,"label":"2007\nemissions peak","y":0.98},
          {"year":2020,"label":"2020\nCOVID","y":0.62}]
INTRO = ("For two centuries the United States built its economy on fossil fuels. "
         "Coal powered the first industrial century; oil and gas took over after 1945. "
         "Emissions peaked in 2007 and have fallen since — yet the US still emits far "
         "above the world average and remains the largest single contributor to the "
         "CO\u2082 humanity has ever released.")
FOOTER = "Source: Global Carbon Budget (2025) via Our World in Data (CC BY).  Built with viz_lib."

def _load_hero():
    d = load_dataset("us_co2_by_fuel")
    for f in FUELS:
        d[f] = pd.to_numeric(d[f], errors="coerce") / 1e9
    return d

def _load_share():
    rows = []
    for fuel, name in [("Oil","share_cumulative_oil"),("Coal","share_cumulative_coal"),
                       ("Cement","share_cumulative_cement")]:
        dd = load_dataset(name)
        col = [c for c in dd.columns if "Share" in c][0]
        rows.append({"Fuel": fuel,
                     "share": dd[(dd.Entity=="United States") & (dd.Year==2024)][col].iloc[0]})
    return pd.DataFrame(rows)

def _draw_ghg(ax):
    g = load_dataset("us_percapita_ghg")
    col = [c for c in g.columns if "greenhouse" in c.lower()][0]
    x, y = g.Year.to_numpy(), g[col].to_numpy()
    c = series_color(0)
    ax.plot(x, y, color=c, linewidth=2.2, solid_capstyle="round")
    ax.fill_between(x, y, color=c, alpha=0.08)
    ax.set_ylim(0, y.max()*1.08); ax.set_xlim(x.min(), x.max())
    ax.annotate(f"{y[-1]:.0f} t", xy=(x[-1], y[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontweight="bold", color=c)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    ax.tick_params(length=0)
    ax.set_title("Even as CO\u2082 fell, the US still emits ~18 t per person",
                 loc="left", fontsize=13, fontweight="bold", pad=16)

apply_theme()
fig = plt.figure(figsize=(12, 15))
gs = fig.add_gridspec(2, 2, height_ratios=[1.45, 1.0], top=0.78, bottom=0.06,
                      left=0.08, right=0.95, hspace=0.42, wspace=0.28)
fig.text(0.08, 0.965, "A CENTURY OF AMERICAN CARBON", fontsize=30, fontweight="bold",
         ha="left", va="top", color="#0b0b0b")
fig.text(0.08, 0.925, INTRO, fontsize=12.5, ha="left", va="top", color="#52514e", wrap=True)
fig.text(0.08, 0.025, FOOTER, fontsize=9, ha="left", va="bottom", color="#898781")

stacked_area(_load_hero(), x="Year", series=FUELS, ax=fig.add_subplot(gs[0, :]),
             y_label="Billion tonnes CO\u2082 / year", title="Coal gave way to oil and gas",
             subtitle="US CO\u2082 emissions by fuel or industry, 1800\u20132024", events=EVENTS)
_draw_ghg(fig.add_subplot(gs[1, 0]))
ranked_bar(_load_share(), category="Fuel", value="share", value_fmt="{:.0f}%",
           ax=fig.add_subplot(gs[1, 1]),
           title="A quarter of all oil CO\u2082 ever, from one country",
           subtitle="US share of the world's cumulative CO\u2082, by source")
fig

---
*Built with `viz_lib` — a small plotting library (functions embedded above).*  
Data: Global Carbon Budget (2025) via Our World in Data (CC BY).